In [ ]:
# Import libraries and data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
import json
import warnings
warnings.filterwarnings("ignore")
# Set the style for seaborn
sns.set(style="whitegrid")

narrative_analysis_df = pd.read_csv('../data/data/narrative_analysis_id_attitude_event.csv')
narrative_analysis_df = narrative_analysis_df.drop_duplicates(subset=['post_id'], keep='first')
readable_output = '../data/data/narrative_analysis_id_root_post_annotated.csv'
import pandas as pd
import ast
posts = []
predictions = []

with open(readable_output, 'r', encoding='utf-8') as f:
    lines = f.readlines()
    for i in range(0, len(lines), 3):  # every post/prediction separated by 2 newlines
        post_line = lines[i].strip()
        prediction_line = lines[i+1].strip()
        
        post = post_line.replace("Post: ", "")
        prediction = prediction_line.replace("Prediction: ", "")
        
        posts.append(post)
        predictions.append(prediction)

df = pd.DataFrame({'post_full_content': posts, 'prediction': predictions})

def clean_and_parse_prediction(prediction_str):
    # Remove "Prediction: " prefix if it exists
    if isinstance(prediction_str, str) and prediction_str.startswith("Prediction:"):
        prediction_str = prediction_str.replace("Prediction:", "").strip()
    
    try:
        parsed = ast.literal_eval(prediction_str)
    except Exception:
        parsed = []

    # Flatten if parsed is a nested list (just in case)
    if isinstance(parsed, list) and len(parsed) == 1 and isinstance(parsed[0], list):
        parsed = parsed[0]

    result = {'trump': None, 'biden': None, 'other': None}
    for item in parsed:
        if not isinstance(item, dict):
            continue
        target = item.get('target', '').lower()
        attitude = item.get('attitude', None)
        if target in result:
            result[target] = attitude
        else:
            result['other'] = attitude
    return pd.Series(result)

# Apply the fix
df[['trump', 'biden', 'other']] = df['prediction'].apply(clean_and_parse_prediction)

#Reply data
NDIR_df = pd.read_csv('../data/data/NDIR_df.csv', )


In [ ]:
with open("../narrative_analysis_output/components_with_conversation_and_classification_data.json", "r") as f:
    NDIR = json.load(f)
with open("../narrative_analysis_output_1000/components_with_conversation_and_classification_data.json", "r") as f:
    NDIR1000 = json.load(f)

NDIR = NDIR + NDIR1000
NDIR_df = []
import re
pattern = r'"classification":\s*"(\w+)"'

for component in NDIR:
    component_id = component['component_id']
    for platform in component['platforms'].keys():
        for root_post in component['platforms'][platform]['root_posts']:
            root_post_id = root_post['id']
            for reply in root_post['replies']:
                reply_content = reply['content']
                reply_createdat = reply['created_at']
                reply_id = reply['id']
                if 'classification' in reply and 'raw_response' in reply['classification']:
                    # Extract classification from raw_response using regex
                    text = reply['classification']['raw_response']
                    match = re.search(pattern, text)
                    if match:
                        classification = match.group(1).lower()
                    else:
                        classification = None
                else:
                    classification = reply.get('classification', {}).get('classification', None)
                post = {
                    'component_id': component_id,
                    'platform': platform,
                    'root_post_id': root_post_id,
                    'reply_content': reply_content,
                    'reply_createdat': reply_createdat,
                    'reply_id': reply_id,
                    'classification': classification
                }
                NDIR_df.append(post)

NDIR_df = pd.DataFrame(NDIR_df)

In [ ]:
full_df = NDIR_df.merge(narrative_analysis_df,left_on="root_post_id", right_on='post_id', suffixes=('_reply', '_root_post'),)

# Plot descriptive statistics

## Radar for NDI 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
narrative_analysis_df = pd.read_csv("../data/data/narrative_analysis_id_root_post_full.csv")
narrative_analysis_df = narrative_analysis_df.loc[narrative_analysis_df['post_full_content'].str.contains(r"(^|[^A-Za-z0-9_])(trump|biden|hunter)($|[^A-Za-z0-9_])", case=False, na=False), ]
# 1. Merge narrative vectors with component NDI data
merged = narrative_analysis_df



# 3. Average vectors for each platform
narrative_frames = [
    "Persecution", "Corruption", "Accountability", "Irony/Detachment",
    "Heroism", "Civic Critique", "Moral Decay", "Media Manipulation",
    "Strategic Pragmatism", "Cultural Identity"
]

def compute_avg_vectors(df):
    return df.groupby('platform')[narrative_frames].mean()


def plot_radar(avg_df, title):
    categories = narrative_frames
    labels = avg_df.index.tolist()
    num_vars = len(categories)

    angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
    angles += angles[:1]
    legend_labels = {
        'truth': 'Truth Social',
        'bluesky': 'BlueSky',
        'mastodon': 'Mastodon'
    }
 

    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True), dpi=400)

    for label in labels:
        values = avg_df.loc[label].tolist()
        values += values[:1]
        label = legend_labels.get(label, label)
        ax.plot(angles, values, label=label, linewidth=2)
        ax.fill(angles, values, alpha=0.1)

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)

    # Use regular label placement with more padding
    ax.set_thetagrids(
        np.degrees(angles[:-1]),
        labels=categories,
        fontsize=11
    )

    for label, angle in zip(ax.get_xticklabels(), angles[:-1]):
        angle_deg = np.degrees(angle)
        if 216 <= angle_deg:
            label.set_rotation(angle_deg + 180)
            label.set_horizontalalignment('right')
        else:
            label.set_rotation(angle_deg)
            label.set_horizontalalignment('left')
        label.set_verticalalignment('center')
        label.set_rotation_mode('anchor')
        label.set_fontsize(16)
        label.set_position((1.08, 0))  # slightly padded out

    ax.set_title(title, y=1.12, fontsize=16)
    min_value = avg_df.min().min()
    max_value = avg_df.max().max()
    yticks = np.linspace(min_value, max_value, num=5)
    ax.set_yticks(yticks)
    ax.set_yticklabels([f"{y:.2f}" for y in yticks], color='gray', fontsize=12)
    
    ax.legend(loc='right', bbox_to_anchor=(1.3, 0.5), fontsize=11)

    plt.tight_layout()
    plt.show()


plot_radar(narrative_analysis_df.groupby('platform')[narrative_frames].mean(), "Overall Platform Narrative Profiles")


In [ ]:
debate = narrative_analysis_df.loc[narrative_analysis_df['presidential_election_debate'],]

In [ ]:
debate.to_csv('../data/external/debate.csv', index=False)

In [ ]:
debate.columns

## Bar plot for NDI-R

In [ ]:
# First, shuffle the full DataFrame
df_shuffled = full_df.sample(frac=1, random_state=42)

# Then, drop duplicates based on root_post_id — keeps the first occurrence
df_no_duplicates = df_shuffled.drop_duplicates(subset='root_post_id')

# Finally, sample 200 from the deduplicated set
sampled_df = df_no_duplicates.sample(n=200, random_state=42)

# (Optional) Reset index
sampled_df = sampled_df.reset_index(drop=True)



In [ ]:
sampled_df.to_csv("../data/data/NDIR_sampled_200.csv",index=False)

In [ ]:
sampled_df[['post_full_content','reply_content']].to_csv('../data/data/NDIR_sampled_200_gpt.csv', index=False)

In [ ]:
from vllm import LLM
import json_repair
pipe = LLM(
                model="google/gemma-3-27b-it",
                tensor_parallel_size=1,
                gpu_memory_utilization=0.95,
                max_model_len=8192,
            )



In [ ]:

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-27b-it")
def analyze_reply_chain( text: str ):

    prompt = (
        "For the following post, identify all political figures or entities discussed and the attitude expressed toward each. "
        "Only consider three possible targets: 'trump', 'biden', or 'other'. For each target, determine whether the post expresses a "
        "'positive', 'negative', or 'neutral' attitude.\n\n"

        "Instructions:\n"
        "- Resolve indirect or implicit references, including pronouns (e.g., 'he', 'our president'), nicknames, emojis, or known context.\n"
        "- Sarcasm, irony, or emotionally charged language should be interpreted for actual sentiment, not literal phrasing.\n"
        "- If a post mentions a 'president's birthday' or celebrates 'our favorite president' in June, it should be interpreted as referring to Donald Trump, whose birthday is June 14.\n"
        "- Do not assume sentiment is positive just because a figure is mentioned in a favorable context (e.g., 'trying to protect Biden' in a sentence full of anger is likely still anti-Biden).\n"
        "- A single post may express opinions about multiple figures. Each should be included with its own sentiment.\n\n"

        "Output format: Return a JSON list of objects like: "
        "[{\"target\": \"biden\", \"attitude\": \"negative\"}, {\"target\": \"trump\", \"attitude\": \"positive\"}]\n\n"

        f"Post:\n{text}"
    )

    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": prompt
        },
    ]
    from vllm import SamplingParams
    params = SamplingParams(
        temperature=0.7,
        top_p=0.9,
        max_tokens=512
    )
    messages = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

    output = pipe.generate(messages, sampling_params=params)
    
    response_text = output[0].outputs[0].text.strip().lower()
        
    # Validate response is one of the expected categories
    results = json_repair.loads(response_text)
    output = {}
    for result in results:
        if result['target'] == 'trump':
            output['trump'] = result['attitude']
        elif result['target'] == 'biden':
            output['biden'] = result['attitude']
        else:
            output['other'] = result['attitude']

    return output


In [ ]:
need_todo = narrative_analysis_df.loc[(narrative_analysis_df['trump'].isna()) & (narrative_analysis_df['biden'].isna())& (narrative_analysis_df['other'].isna())]

In [ ]:
index

In [ ]:
need_todo.loc[need_todo.index >= index]

In [ ]:
%%capture
from tqdm.auto import tqdm
start_index = index

for index, row in tqdm(need_todo.loc[need_todo.index >= start_index].iterrows()):
    text = row['post_full_content']
    analysis_result = analyze_reply_chain(text)
    narrative_analysis_df.at[index, 'trump'] = analysis_result.get('trump', None)
    narrative_analysis_df.at[index, 'biden'] = analysis_result.get('biden', None)
    narrative_analysis_df.at[index, 'other'] = analysis_result.get('other', None)

In [ ]:
narrative_analysis_df.loc[(narrative_analysis_df['trump'].isna()) & (narrative_analysis_df['biden'].isna())& (narrative_analysis_df['other'].isna())]

In [ ]:
narrative_analysis_df.to_csv('../data/data/narrative_analysis_id_attitude_event.csv', index=False)

In [ ]:

NDIR_df = NDIR_df[['root_post_id','classification','reply_content']].merge(df.drop_duplicates(subset='post_id'), left_on='root_post_id', right_on='post_id', how='left')

In [ ]:
stack = NDIR_df.groupby('platform')['classification'].value_counts().reset_index(name='count')
#convert count to percentage for each platform
stack['percentage'] = (
    stack['count'] / stack.groupby('platform')['count'].transform('sum') * 100
)
#stacked bar plot
import matplotlib.pyplot as plt
import pandas as pd

# Pivot your data so classifications become columns
stacked = stack.pivot_table(
    index="platform",
    columns="classification",
    values="percentage",
    aggfunc="sum"
)

# Plot stacked bar chart
fig, ax = plt.subplots(figsize=(8, 6), dpi=400)
stacked.plot(kind="bar", stacked=True, ax=ax)
# Annotate each segment
for container in ax.containers:
    # Optional: only label segments that are non-zero
    ax.bar_label(container, fmt="%.1f%%", label_type="center", color="white", fontsize=10)

# Beautify
ax.set_ylabel("Percentage")
ax.set_xlabel("Platform", fontweight='bold')
ax.set_xticklabels(['Bluesky', "Mastodon", "Truth Social"], rotation=0, fontsize=12)
#ax.set_title("Stacked Bar Plot of Classification by Platform")
ax.legend(title="Classification", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
# Step 2: Expand multi-label target-attitude column
def get_target_attitudes(row):
    tags = []
    if row['trump'] == 'positive':
        tags.append("Trump++")
    elif row['trump'] == 'negative':
        tags.append("Trump--")
    if row['biden'] == 'positive':
        tags.append("Biden++")
    elif row['biden'] == 'negative':
        tags.append("Biden--")
    return tags if tags else ['Neutral']

df['target_attitudes'] = df.apply(get_target_attitudes, axis=1)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Step 1: Group and count
grouped = NDIR_df.groupby(['platform', 'classification', ]).size().reset_index(name='count')



# Step 5: Pivot and normalize to percent
pivot = grouped.pivot(index='platform', columns='classification', values='count').fillna(0)
pivot_percent = pivot.div(pivot.sum(axis=1), axis=0) * 100  # Normalize to percent

import seaborn as sns
plt.figure(figsize=(12, 8), dpi=300)
sns.heatmap(pivot_percent.T, annot=True, cbar_kws={'label': 'Percentage'})
#plt.title('Attitude Distribution Heatmap by Platform', fontsize=18, fontweight='bold')
plt.ylabel('Attitude', fontsize=18, fontweight='bold')
plt.xlabel('Platform', fontsize=18, fontweight='bold')

plt.xticks(ticks=np.arange(len(pivot.index)) + 0.5, labels=["BlueSky", 'Mastodon', 'TruthSocial'], fontsize=18)
plt.yticks(fontsize=18)
plt.tight_layout()
plt.show()




# Step 1: Calculate the NDI

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import jensenshannon
from scipy.stats import zscore
from itertools import combinations

def assign_ideology(row):
    if row['trump'] == 'positive':
        return -1  # right-leaning
    elif row['biden'] == 'negative':
        return -0.5
    elif row['trump'] == 'negative':
        return 0.5
    elif row['biden'] == 'positive':
        return 1   # left-leaning
    else:
        return 0   # neutral / unclear

def calculate_ideology_weighted_ndi_jsd_3platform(df):
    narrative_frames = [
        "Persecution", "Corruption", "Accountability", "Irony/Detachment",
        "Heroism", "Civic Critique", "Moral Decay", "Media Manipulation",
        "Strategic Pragmatism", "Cultural Identity"
    ]

    df = df.copy()

    # Assign ideology scores
    df['ideology'] = df.apply(assign_ideology, axis=1)

    results = []

    for component_id in df['component_id'].unique():
        comp_df = df[df['component_id'] == component_id]
        platforms = comp_df['platform'].unique()
        if len(platforms) < 2:
            continue

        vectors = {}

        for platform in platforms:
            plat_df = comp_df[comp_df['platform'] == platform]
            if plat_df.empty:
                continue

            weighted = []
            for _, row in plat_df.iterrows():
                vec = np.array([row[frame] for frame in narrative_frames])
                weighted.append(row['ideology'] * vec)

            vec_sum = np.sum(weighted, axis=0)
            if vec_sum.sum() != 0:
                vec_norm = vec_sum / (np.sum(np.abs(vec_sum)) + 1e-9)
                vec_norm = np.clip(vec_norm, 0, None)  # remove negatives
                if vec_norm.sum() > 0:
                    vec_norm /= vec_norm.sum()
                    vectors[platform] = vec_norm

        # Compute pairwise JSD
        jsds = []
        for a, b in combinations(vectors.keys(), 2):
            jsd = jensenshannon(vectors[a], vectors[b], base=2) ** 2
            jsds.append(jsd)

        avg_jsd = np.mean(jsds) if jsds else 0.0
        results.append({
            'component_id': component_id,
            'NDI_POLAR_JSD': avg_jsd,
            'platforms_present': list(vectors.keys())
        })

    result_df = pd.DataFrame(results)
    result_df['NDI_POLAR_JSD_z'] = zscore(result_df['NDI_POLAR_JSD'])
    return result_df


# Example usage
ndi_df = calculate_ideology_weighted_ndi_jsd_3platform(narrative_analysis_df)
print(f"Calculated NDI-TW for {len(ndi_df)} components")
print(f"NDI-TW range: {ndi_df['NDI_POLAR_JSD'].min():.4f} to {ndi_df['NDI_POLAR_JSD'].max():.4f}")
print(f"Mean NDI-TW: {ndi_df['NDI_POLAR_JSD'].mean():.4f}")



In [ ]:

import numpy as np
import pandas as pd
from scipy.spatial.distance import jensenshannon
from scipy.stats import zscore

def assign_ideology(row):
    if row['trump'] == 'positive':
        return -1  # pro-Trump
    elif row['biden'] == 'negative':
        return -0.5
    elif row['trump'] == 'negative':
        return 0.5
    elif row['biden'] == 'positive':
        return 1
    else:
        return 0  # neutral or unclear

def calculate_ideology_weighted_ndi_jsd(df):
    narrative_frames = [
        "Persecution", "Corruption", "Accountability", "Irony/Detachment",
        "Heroism", "Civic Critique", "Moral Decay", "Media Manipulation",
        "Strategic Pragmatism", "Cultural Identity"
    ]

    df = df.copy()

    # Platform group mapping
    def group_platform(p):
        p = p.lower()
        if 'truth' in p:
            return 'Truth'
        elif 'bluesky' in p or 'mastodon' in p:
            return 'Bluesky_Mastodon'
        else:
            return None

    df['platform_group'] = df['platform'].apply(group_platform)
    df = df[df['platform_group'].notna()]
    df['ideology'] = df.apply(assign_ideology, axis=1)

    results = []

    for component_id in df['component_id'].unique():
        comp_df = df[df['component_id'] == component_id]

        vectors = {}
        for group in ['Truth', 'Bluesky_Mastodon']:
            group_df = comp_df[comp_df['platform_group'] == group]
            if group_df.empty:
                continue

            weighted = []
            for _, row in group_df.iterrows():
                vec = np.array([row[frame] for frame in narrative_frames])
                weighted.append(row['ideology'] * vec)

            vec_sum = np.sum(weighted, axis=0)
            if vec_sum.sum() != 0:
                vec_norm = vec_sum / (np.sum(np.abs(vec_sum)) + 1e-9)
                vec_norm = np.clip(vec_norm, 0, None)
                if vec_norm.sum() > 0:
                    vec_norm /= vec_norm.sum()
                    vectors[group] = vec_norm

        # Fill missing group with uniform distribution
        if 'Truth' not in vectors and 'Bluesky_Mastodon' in vectors:
            dim = len(narrative_frames)
            vectors['Truth'] = np.ones(dim) / dim
        elif 'Bluesky_Mastodon' not in vectors and 'Truth' in vectors:
            dim = len(narrative_frames)
            vectors['Bluesky_Mastodon'] = np.ones(dim) / dim

        if len(vectors) == 2:
            jsd = jensenshannon(vectors['Truth'], vectors['Bluesky_Mastodon'], base=2) ** 2
            results.append({
                'component_id': component_id,
                'NDI_POLAR_JSD': jsd,
                'has_truth': 'Truth' in comp_df['platform_group'].unique(),
                'has_bluesky_mastodon': 'Bluesky_Mastodon' in comp_df['platform_group'].unique()
            })

    result_df = pd.DataFrame(results)
    result_df['NDI_POLAR_JSD_z'] = zscore(result_df['NDI_POLAR_JSD'])
    return result_df

ndi_truth_df = calculate_ideology_weighted_ndi_jsd(narrative_analysis_df)
print(f"Calculated NDI-TW for {len(ndi_truth_df)} components")
print(f"NDI-TW range: {ndi_truth_df['NDI_POLAR_JSD'].min():.4f} to {ndi_truth_df['NDI_POLAR_JSD'].max():.4f}")
print(f"Mean NDI-TW: {ndi_truth_df['NDI_POLAR_JSD'].mean():.4f}")

In [ ]:
# Re-import necessary libraries after kernel reset
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Placeholder: define plot function and wait for data to be reloaded
def plot_ndi_op_distribution(df):
    plt.figure(figsize=(8, 5), dpi=300)
    sns.histplot(df['NDI_POLAR_JSD'], bins=40, kde=True, color='#6276aa', edgecolor='black')
    mean_val = df['NDI_POLAR_JSD'].mean()
    plt.axvline(mean_val, color='#cd6065', linestyle='--', linewidth=1.5, label=f'Mean = {mean_val:.3f}')
    plt.xlabel('NDI-OP Score', fontsize=18, fontweight='bold')
    plt.ylabel('Component Count', fontsize=18, fontweight='bold')
    #plt.title('Distribution of Narrative Divergence Index (NDI-TW)', fontsize=14)
    plt.xticks(fontsize=16)
    plt.yticks(fontsize=16)
    plt.legend()
    plt.tight_layout()
    plt.show()

"Ready for data upload to draw NDI-OP distribution."
plot_ndi_op_distribution(ndi_df)# Re-import necessary libraries after kernel reset
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Placeholder: define plot function and wait for data to be reloaded
def plot_ndi_op_distribution(df):
    plt.figure(figsize=(8, 5), dpi=300)
    sns.histplot(df['NDI_POLAR_JSD'], bins=40, kde=True, color='#6276aa', edgecolor='black')
    mean_val = df['NDI_POLAR_JSD'].mean()
    plt.axvline(mean_val, color='#cd6065', linestyle='--', linewidth=1.5, label=f'Mean = {mean_val:.3f}')
    plt.xlabel('NDI-TW Score', fontsize=18, fontweight='bold')
    plt.ylabel('Component Count', fontsize=18, fontweight='bold')
    #plt.title('Distribution of Narrative Divergence Index (NDI-TW)', fontsize=14)
    plt.xticks(fontsize=16)
    plt.yticks(fontsize=16)
    plt.legend()
    plt.tight_layout()
    plt.show()

"Ready for data upload to draw NDI-OP distribution."
plot_ndi_op_distribution(ndi_truth_df)

## Heatmap of Frame-Target-Attitude

In [ ]:
summary

In [ ]:
import seaborn as sns
narrative_frames = [
    "Persecution", "Corruption", "Accountability", "Irony/Detachment",
    "Heroism", "Civic Critique", "Moral Decay", "Media Manipulation",
    "Strategic Pragmatism", "Cultural Identity"
]
df = narrative_analysis_df.loc[narrative_analysis_df['in_reply_to_id'].isna(),:]
df_long = df.melt(
    id_vars=['platform', 'biden', 'trump', 'other'],
    value_vars=narrative_frames,
    var_name='frame',
    value_name='frame_score'
)
def get_target_attitude(row):
    for target in ['biden', 'trump', 'other']:
        att = row[target]
        if att in ['positive', 'negative', 'neutral']:
            return f"{target.capitalize()}-{att.capitalize()}"
    return "Unknown"

df_long['target_attitude'] = df_long.apply(get_target_attitude, axis=1)
df_long['platform_group'] = df_long['platform'].replace({'bluesky': 'BlueSky and Mastodon', 'mastodon': 'BlueSky and Mastodon', 'truth': 'Truth Social'})
df = df_long.copy()
# Filter to only include Trump and Biden targets
target_whitelist = [
    'Biden-Negative', 'Biden-Neutral', 'Biden-Positive',
    'Trump-Negative', 'Trump-Neutral', 'Trump-Positive'
]
filtered = df[df['target_attitude'].isin(target_whitelist)]

# Pivot and normalize per platform group
pivoted = (
    filtered
    .groupby(['platform_group', 'frame', 'target_attitude'])['frame_score']
    .sum()
    .reset_index()
    .pivot_table(index=['platform_group', 'frame'], columns='target_attitude', values='frame_score', fill_value=0)
)

# Normalize rows within each platform group
pivoted_norm = pivoted.groupby(level=0).apply(lambda x: x.div(x.sum(axis=1), axis=0))

# Plot heatmaps for each platform group
platforms = pivoted_norm.index.get_level_values(0).unique()
fig, axes = plt.subplots(1, len(platforms), figsize=(18, 6), sharey=True, dpi=300)

for i, platform in enumerate(platforms):
    sns.heatmap(
        pivoted_norm.loc[platform],
        annot=True, fmt=".2f", 
        ax=axes[i], cbar=i == len(platforms) - 1
    )
    axes[i].set_title(f"{platform}", fontsize=20, fontweight='bold')
    axes[i].set_xlabel("Target–Attitude", fontsize=18, fontweight='bold')
    axes[i].set_ylabel("Narrative Frame" if i == 0 else "", fontsize=16, fontweight='bold')
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45, ha='right', fontsize=18)
axes[0].set_yticklabels(narrative_frames, rotation=0, ha='right', fontsize=16)
#plt.suptitle("Frame × Target–Attitude Distribution (Row-normalized)", fontsize=22, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
import seaborn as sns
narrative_frames = [
    "Persecution", "Corruption", "Accountability", "Irony/Detachment",
    "Heroism", "Civic Critique", "Moral Decay", "Media Manipulation",
    "Strategic Pragmatism", "Cultural Identity"
]
df = narrative_analysis_df.loc[narrative_analysis_df['in_reply_to_id'].isna(),:].drop_duplicates(subset=['post_id'])

# ——前面的 melt 保持不变——
df_long = df.melt(
    id_vars=['platform', 'biden', 'trump', 'other'],
    value_vars=narrative_frames,
    var_name='frame',
    value_name='frame_score'
)

# 收集多个 target-attitude（而不是返回第一个）
def get_target_attitudes(row):
    out = []
    for target in ['biden', 'trump', 'other']:
        att = str(row[target]).strip().lower()
        if att in {'positive', 'negative', 'neutral'}:
            out.append(f"{target.capitalize()}-{att.capitalize()}")
    return out if out else ['Unknown']

# 生成列表后按列表展开成多行
df_long = (
    df_long
    .assign(target_attitude_list=df_long.apply(get_target_attitudes, axis=1))
    .explode('target_attitude_list')
    .rename(columns={'target_attitude_list': 'target_attitude'})
)

# 合并平台分组
df_long['platform_group'] = df_long['platform'].replace({
    'bluesky': 'BlueSky and Mastodon',
    'mastodon': 'BlueSky and Mastodon',
    'truth': 'Truth Social'
})

# 之后的流程无需大改：白名单筛选、聚合、归一化、绘图
target_whitelist = [
    'Biden-Negative', 'Biden-Neutral', 'Biden-Positive',
    'Trump-Negative', 'Trump-Neutral', 'Trump-Positive'
]
filtered = df_long[df_long['target_attitude'].isin(target_whitelist)]

pivoted = (
    filtered
    .groupby(['platform_group', 'frame', 'target_attitude'])['frame_score']
    .sum()
    .reset_index()
    .pivot_table(index=['platform_group', 'frame'],
                 columns='target_attitude',
                 values='frame_score',
                 fill_value=0)
)

# 确保列顺序一致（可选）
pivoted = pivoted.reindex(columns=target_whitelist, fill_value=0)

# 每个平台组内做行归一化
pivoted_norm = pivoted.groupby(level=0).apply(lambda x: x.div(x.sum(axis=1), axis=0))

# ——后面的绘图保持不变——
platforms = pivoted_norm.index.get_level_values(0).unique()
fig, axes = plt.subplots(1, len(platforms), figsize=(18, 6), sharey=True, dpi=300)
for i, platform in enumerate(platforms):
    sns.heatmap(
        pivoted_norm.loc[platform],
        annot=True, fmt=".2f",
        ax=axes[i], cbar=i == len(platforms) - 1
    )
    axes[i].set_title(f"{platform}", fontsize=20, fontweight='bold')
    axes[i].set_xlabel("Target–Attitude", fontsize=18, fontweight='bold')
    axes[i].set_ylabel("Narrative Frame" if i == 0 else "", fontsize=16, fontweight='bold')
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45, ha='right', fontsize=18)

axes[0].set_yticklabels(narrative_frames, rotation=0, ha='right', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

narrative_frames = [
    "Persecution", "Corruption", "Accountability", "Irony/Detachment",
    "Heroism", "Civic Critique", "Moral Decay", "Media Manipulation",
    "Strategic Pragmatism", "Cultural Identity"
]
df = narrative_analysis_df.loc[narrative_analysis_df['in_reply_to_id'].isna(),:].drop_duplicates(subset=['post_id'])

# ——前面的 melt 保持不变——
df_long = df.melt(
    id_vars=['platform', 'biden', 'trump', 'other'],
    value_vars=narrative_frames,
    var_name='frame',
    value_name='frame_score'
)

def get_target_attitudes(row):
    out = []
    for target in ['biden', 'trump', 'other']:
        att = str(row[target]).strip().lower()
        if att in {'positive', 'negative', 'neutral'}:
            out.append(f"{target.capitalize()}-{att.capitalize()}")
    return out if out else ['Unknown']

df_long = (
    df_long
    .assign(target_attitude_list=df_long.apply(get_target_attitudes, axis=1))
    .explode('target_attitude_list')
    .rename(columns={'target_attitude_list': 'target_attitude'})
)

df_long['platform_group'] = df_long['platform'].replace({
    'bluesky': 'BlueSky and Mastodon',
    'mastodon': 'BlueSky and Mastodon',
    'truth': 'Truth Social'
})

target_whitelist = [
    'Biden-Negative', 'Biden-Neutral', 'Biden-Positive',
    'Trump-Negative', 'Trump-Neutral', 'Trump-Positive'
]
filtered = df_long[df_long['target_attitude'].isin(target_whitelist)]

pivoted = (
    filtered
    .groupby(['platform_group', 'frame', 'target_attitude'])['frame_score']
    .sum()
    .reset_index()
    .pivot_table(index=['platform_group', 'frame'],
                 columns='target_attitude',
                 values='frame_score',
                 fill_value=0)
)

# 确保列顺序一致（可选）
pivoted = pivoted.reindex(columns=target_whitelist, fill_value=0)

# ====== 关键改动：每个平台内“按列归一化” -> P(Frame | Target-Attitude) ======
# 让每一列（一个 target-attitude）在各 frame 上的和为 1
pivoted_norm = (
    pivoted
    .groupby(level=0)
    .apply(lambda x: x.div(x.sum(axis=0).replace(0, np.nan), axis=1))
    .fillna(0)
)

# （可选）过滤样本量太小的 target-attitude 列，避免噪声
# 例：要求每个平台该列的总“分数”>= min_total（你也可以换成贴文数来判定）
# min_total = 10.0
# keep_cols = []
# for col in pivoted.columns:
#     totals_by_platform = pivoted.xs('BlueSky and Mastodon').sum(axis=0)[col], pivoted.xs('Truth Social').sum(axis=0)[col]
#     if all(t >= min_total for t in totals_by_platform):
#         keep_cols.append(col)
# if keep_cols:
#     pivoted_norm = pivoted_norm[keep_cols]

# ——后面的绘图保持不变——
platforms = pivoted_norm.index.get_level_values(0).unique()
fig, axes = plt.subplots(1, len(platforms), figsize=(18, 6), sharey=True, dpi=300)
for i, platform in enumerate(platforms):
    sns.heatmap(
        pivoted_norm.loc[platform],
        annot=True, fmt=".2f",
        ax=axes[i], cbar=i == len(platforms) - 1
    )
    axes[i].set_title(f"{platform}", fontsize=20, fontweight='bold')
    axes[i].set_xlabel("Target–Attitude", fontsize=18, fontweight='bold')
    axes[i].set_ylabel("Narrative Frame" if i == 0 else "", fontsize=16, fontweight='bold')
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45, ha='right', fontsize=18)

axes[0].set_yticklabels(narrative_frames, rotation=0, ha='right', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


# Step 3: Model

In [ ]:
df.loc[df['target_attitude'] == 'Biden++',] 

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

# Step 1: Define narrative frames
narrative_frames = [
    "Persecution", "Corruption", "Accountability", "Irony",
    "Heroism", "Civic Critique", "Moral Decay", "Media Manipulation",
    "Strategic Pragmatism", "Cultural Identity"
]

# Step 2: Copy and rename for safe modeling
df = narrative_analysis_df.loc[narrative_analysis_df['in_reply_to_id'].isna(), :].copy()
df.rename(columns={"Irony/Detachment": "Irony"}, inplace=True)
df.rename(columns={f: f.replace(" ", "_") for f in narrative_frames}, inplace=True)
narrative_frames = [f.replace(" ", "_") for f in narrative_frames]

# Step 3: Add dominant target-attitude variable
def get_target_attitude(row):
    for target in ['trump', 'biden']:
        if row[target] in ['positive', 'negative']:
            return f"{target.capitalize()}{'++' if row[target]=='positive' else '--'}"
    return 'Neutral'

df['target_attitude'] = df.apply(get_target_attitude, axis=1)

# Step 4: Scale frame values
scaler = StandardScaler()
df[narrative_frames] = scaler.fit_transform(df[narrative_frames])

# Step 5: Fit platform-specific model
def fit_platform_model(df, platform_name, outcome_var):
    # Filter platform
    if platform_name == 'truth':
        platform_df = df[df['platform'] == platform_name].copy()
    else:
        platform_df = df[df['platform'] != 'truth'].copy()

    # Keep relevant variables
    platform_df = platform_df[['post_id', 'target_attitude', outcome_var] + narrative_frames].dropna()

    # Collapse to post level
    platform_df = platform_df.groupby(['post_id', 'target_attitude']).mean().reset_index()

    # Create interaction formula with Neutral as reference
    frame_terms = " + ".join(narrative_frames)
    formula = f"{outcome_var} ~ C(target_attitude, Treatment(reference='Neutral')) + ({frame_terms}) "

    model = smf.glm(
        formula=formula,
        data=platform_df,
        family=sm.families.Poisson()
    ).fit()

    return model


# Step 6: Run for each platform
platforms = narrative_analysis_df['platform'].unique()
models = {}
for platform in platforms:
    models[platform] = fit_platform_model(df, platform, 'reblogs_count')

# Step 7: Output summaries
for platform, model in models.items():
    print(f"\nModel Summary for {platform}:\n")
    print(model.summary())
    print("\n" + "="*80 + "\n")



In [ ]:
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

# -------------------------------
# Step 1: Define narrative frames
# -------------------------------
narrative_frames = [
    "Persecution", "Corruption", "Accountability", "Irony",
    "Heroism", "Civic Critique", "Moral Decay", "Media Manipulation",
    "Strategic Pragmatism", "Cultural Identity"
]

# -----------------------------------------------
# Step 2: Copy and rename for safe, consistent cols
# -----------------------------------------------
df = narrative_analysis_df.loc[narrative_analysis_df['in_reply_to_id'].isna(), :].copy()
df.rename(columns={"Irony/Detachment": "Irony"}, inplace=True)
df.rename(columns={f: f.replace(" ", "_") for f in narrative_frames}, inplace=True)
narrative_frames = [f.replace(" ", "_") for f in narrative_frames]

# ------------------------------------
# Step 3: Build target_attitude factor
# ------------------------------------
def get_target_attitude(row):
    for target in ['trump', 'biden']:
        if row.get(target) in ['positive', 'negative']:
            return f"{target.capitalize()}{'+' if row[target]=='positive' else '-'}"
    return 'Neutral'

def get_target_attitudes(row):
    tags = []
    if row.get('biden') == 'positive':
        tags.append("Biden+")
    elif row.get('biden') == 'negative':
        tags.append("Biden-")
    if row.get('trump') == 'positive':
        tags.append("Trump+")
    elif row.get('trump') == 'negative':
        tags.append("Trump-")

    return tags if tags else ['Neutral']

df['target_attitudes'] = df.apply(get_target_attitudes, axis=1)
df = df.explode('target_attitudes').rename(columns={'target_attitudes': 'target_attitude'})



# --------------------------------
# Step 4: Scale (z-score) the frames
#   说明：中心化有助于交互项稳定与解释
# --------------------------------
scaler = StandardScaler()
df[narrative_frames] = scaler.fit_transform(df[narrative_frames])

# -------------------------------------------------------
# Helper: build the formula with FE and interactions
#   主效应：frames + target_attitude
#   交互：frame × target_attitude
#   Neutral 作为基类，保证识别
# -------------------------------------------------------
def build_formula(outcome_var):
    frame_terms = " + ".join(narrative_frames)
    # '*' = 主效应 + 交互；Treatment reference 设为 'Neutral'
    formula = (
        f"{outcome_var} ~ ({frame_terms}) * C(target_attitude, "
        f"Treatment(reference='Neutral'))"
    )
    return formula

# -------------------------------------------------------
# Step 5: Fit one model PER platform with interactions
#   - Poisson GLM + robust (HC3) SE；若你要 NB2，可在此替换 family
#   - 可选：聚类稳健 SE（若有 author_id/thread_id）
# -------------------------------------------------------
def fit_platform_model_interactions(df, platform_name, outcome_var,
                                    cov_type='HC3', cluster_col=None):
    # 仅保留该平台
    platform_df = df[df['platform'] == platform_name].copy()

    # 只留必要列
    keep_cols = ['post_id', 'target_attitude', outcome_var] + narrative_frames
    platform_df = platform_df[keep_cols].dropna()

    # 如果一帖多行，这里聚合到帖层面（均值）
    platform_df = platform_df.groupby(['post_id', 'target_attitude'], as_index=False).mean()

    # 拟合
    formula = build_formula(outcome_var)
    model = smf.glm(
        formula=formula,
        data=platform_df,
        family=sm.families.Poisson()
        # 若要 NB2: family=sm.families.NegativeBinomial(alpha=1.0)  # 可微调 alpha
    ).fit(cov_type=cov_type,
          cov_kwds=({"groups": platform_df[cluster_col]} if (cov_type=="cluster" and cluster_col is not None) else None))

    # 过度离散诊断（可选打印）
    pearson_chi2 = model.pearson_chi2
    df_resid = model.df_resid
    od_ratio = pearson_chi2 / df_resid if df_resid > 0 else float('nan')

    return model, {"pearson_chi2": pearson_chi2, "df_resid": df_resid, "overdispersion_ratio": od_ratio}

# -----------------------------------------
# Step 6: Run for each platform (one model)
# -----------------------------------------
platforms = df['platform'].dropna().unique()
models = {}
diagnostics = {}

for platform in platforms:
    m, diag = fit_platform_model_interactions(
        df=df,
        platform_name=platform,
        outcome_var='reblogs_count',
        cov_type='HC3',            # 或 'cluster'
        cluster_col=None           # 例如 'author_id' 或 'thread_id'
    )
    models[platform] = m
    diagnostics[platform] = diag

# -------------------------------------------------
# Step 7: Summaries + joint tests of interactions
#   使用 wald_test_terms() 报告交互项的联合显著性
# -------------------------------------------------
for platform, model in models.items():
    print(f"\nModel Summary for {platform}:\n")
    print(model.summary())
    print("\n[Diagnostics]")
    print(diagnostics[platform])

    # 术语级别的 Wald 检验（主效应/交互分块）
    try:
        wt = model.wald_test_terms(skip_single=False)
        print("\n[Wald test by term]")
        print(wt.summary_frame())

        # 也可以仅筛选交互项（包含冒号的项）
        print("\n[Joint tests — interaction terms only]")
        term_table = wt.summary_frame()
        inter_terms = term_table[term_table.index.str.contains(":")]
        print(inter_terms)
    except Exception as e:
        print("\n[Wald test terms unavailable]", e)

    print("\n" + "="*80 + "\n")



In [ ]:
df[['trump','biden', 'other']].dropna(how='all')

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

# 1) 定义 frame 列
narrative_frames = [
    "Persecution", "Corruption", "Accountability", "Irony",
    "Heroism", "Civic Critique", "Moral Decay", "Media Manipulation",
    "Strategic Pragmatism", "Cultural Identity"
]

# 2) 拷贝 & 统一列名
df = narrative_analysis_df.loc[narrative_analysis_df['in_reply_to_id'].isna(), :].drop_duplicates(subset=['post_id']).copy()
df.rename(columns={"Irony/Detachment": "Irony"}, inplace=True)
df.rename(columns={f: f.replace(" ", "_") for f in narrative_frames}, inplace=True)
narrative_frames = [f.replace(" ", "_") for f in narrative_frames]

# 3) 生成 target_attitude

def get_target_attitudes(row):
    tags = []
    if row.get('biden') == 'positive':
        tags.append("Biden+")
    elif row.get('biden') == 'negative':
        tags.append("Biden-")
    if row.get('trump') == 'positive':
        tags.append("Trump+")
    elif row.get('trump') == 'negative':
        tags.append("Trump-")

    return tags if tags else ['Neutral']

df['target_attitudes'] = df.apply(get_target_attitudes, axis=1)
df = df.explode('target_attitudes').rename(columns={'target_attitudes': 'target_attitude'})

# ============ 3) 平台合并：Truth vs Bsky/Mastodon ============
df['platform_collapsed'] = np.where(
    df['platform'].str.lower().isin(['bluesky', 'mastodon']),
    'Bsky/Mastodon',
    df['platform'].str.lower()
)

# 只保留两类平台
df = df[df['platform_collapsed'].isin(['truth', 'Bsky/Mastodon'])].copy()

# 4) 合并平台
df['platform_collapsed'] = np.where(
    df['platform'].isin(['bluesky', 'mastodon']),
    'Bsky/Mastodon',
    df['platform']
)

# 5) 仅保留关心的平台
df = df[df['platform_collapsed'].isin(['truth', 'Bsky/Mastodon'])].copy()

# # 6) 标准化 frame
# scaler = StandardScaler()
# df[narrative_frames] = scaler.fit_transform(df[narrative_frames])

# 7) 聚合到 post level
df_post = (
    df[['post_id', 'platform_collapsed', 'target_attitude', 'reblogs_count'] + narrative_frames]
    .dropna()
    .groupby(['post_id', 'platform_collapsed', 'target_attitude'], as_index=False)
    .mean()
)

# 8) 拟合函数（含 frame × target_attitude 交互）
def fit_model_with_interactions(frame_df, platform_name, outcome='reblogs_count'):
    data_ = frame_df[frame_df['platform_collapsed'] == platform_name].copy()
    frame_terms = " + ".join(narrative_frames)
    interaction_terms = " + ".join([f"{f}:C(target_attitude, Treatment(reference='Neutral'))" for f in narrative_frames])
    formula = f"{outcome} ~ C(target_attitude, Treatment(reference='Neutral')) + {frame_terms} + {interaction_terms}"
    model = smf.glm(formula=formula, data=data_, family=sm.families.NegativeBinomial()).fit(cov_type='HC3')
    od_ratio = model.pearson_chi2 / model.df_resid if model.df_resid > 0 else np.nan
    return model, {'overdispersion_ratio': od_ratio, 'n': len(data_)}


# 9) 分别拟合
models = {}
diagnostics = {}
for plat in ['truth', 'Bsky/Mastodon']:
    m, diag = fit_model_with_interactions(df_post, plat, outcome='reblogs_count')
    models[plat] = m
    diagnostics[plat] = diag
    print(f"\n=== Model Summary for {plat} ===\n")
    print(m.summary())
    print("\n[Diagnostics]", diag)
    print("\n" + "="*80 + "\n")
# 在第 9 步循环之后，直接收集并标注显著性
import re

def tidy_glm(model, platform_name, narrative_frames):
    """
    返回一个 DataFrame，每行一个参数（含主效应与交互项），包含：
    term, coef, se, z, p, ci_low, ci_high, platform, term_type, frame, group, sig
    """
    params = model.params
    bse    = model.bse
    pvals  = model.pvalues
    zvals  = params / bse
    ci     = model.conf_int()
    ci.columns = ["ci_low", "ci_high"]

    df_coef = (
        pd.DataFrame({
            "term": params.index,
            "coef": params.values,
            "se":   bse.values,
            "z":    zvals.values,
            "p":    pvals.values,
        })
        .join(ci, how="left")
    )
    df_coef["platform"] = platform_name

    # 解析参数名：类型/对应frame/对应group
    def parse_term(t):
        term_type = "intercept" if t == "Intercept" else ("interaction" if ":" in t else "main")
        frame = None
        # 把 statsmodels 里下划线版本的 frame 名称作为匹配
        for f in narrative_frames:
            if t.startswith(f + ":") or t == f:
                frame = f
                break
        m = re.search(r"\[T\.(Biden\+|Biden-|Trump\+|Trump-)\]", t)
        group = m.group(1) if m else None
        return pd.Series({"term_type": term_type, "frame": frame, "group": group})

    parsed = df_coef["term"].apply(parse_term)
    df_coef = pd.concat([df_coef, parsed], axis=1)

    # 显著性星标
    def star(p):
        if p < 0.001: return "***"
        if p < 0.01:  return "**"
        if p < 0.05:  return "*"
        if p < 0.10:  return "·"
        return ""
    df_coef["sig"] = df_coef["p"].apply(star)
    return df_coef

# 收集两个平台的“tidy”系数表
tidy_list = []
for plat in ['truth', 'Bsky/Mastodon']:
    m = models[plat]
    tidy_list.append(tidy_glm(m, plat, narrative_frames))

coef_tidy = pd.concat(tidy_list, ignore_index=True)

# —— 你要“顺便看看交互项有没有显著”，直接筛一下即可：
interaction_sig = (
    coef_tidy
    .query("term_type == 'interaction'")
    .sort_values(["platform", "p"])
)

# 打印每个平台前若干个最显著的交互项
for plat in ['truth', 'Bsky/Mastodon']:
    print(f"\n=== Significant interactions @ {plat} (p < 0.05) ===")
    display(
        interaction_sig
        .query("platform == @plat and p < 0.05")
        .loc[:, ["term","frame","group","coef","se","z","p","ci_low","ci_high","sig"]]
        .head(30)
    )



In [ ]:
sum(df['reblogs_count']<0)

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
import re

# 1) 定义 frame 列（原始名 -> 下划线）
narrative_frames = [
    "Persecution", "Corruption", "Accountability", "Irony",
    "Heroism", "Civic Critique", "Moral Decay", "Media Manipulation",
    "Strategic Pragmatism", "Cultural Identity"
]

# 2) 拷贝 & 统一列名（只保留 root posts；每个 post 一行）
df = (
    narrative_analysis_df
    #.loc[narrative_analysis_df['in_reply_to_id'].isna(), :]
    .drop_duplicates(subset=['post_id'])
    .copy()
)
df.rename(columns={"Irony/Detachment": "Irony"}, inplace=True)
df.rename(columns={f: f.replace(" ", "_") for f in narrative_frames}, inplace=True)
narrative_frames = [f.replace(" ", "_") for f in narrative_frames]

# 3) multi-hot 指示变量（避免符号，用安全列名）
df['BidenPos'] = (df['biden']  == 'positive').astype(int)
df['BidenNeg'] = (df['biden']  == 'negative').astype(int)
df['TrumpPos'] = (df['trump']  == 'positive').astype(int)
df['TrumpNeg'] = (df['trump']  == 'negative').astype(int)
indicators = ['BidenPos','BidenNeg','TrumpPos','TrumpNeg']

# 4) 平台合并：Truth vs Bsky/Mastodon（一次即可）
platform_lower = df['platform'].str.lower()
df['platform_collapsed'] = np.where(
    platform_lower.isin(['bluesky', 'mastodon']),
    'Bsky/Mastodon',
    platform_lower
)
df = df[df['platform_collapsed'].isin(['truth', 'Bsky/Mastodon'])].copy()

# 5) （可选）标准化 frame 分数
scaler = StandardScaler()
df[narrative_frames] = scaler.fit_transform(df[narrative_frames])

# 6) 聚合到 post level（不再按 target 分组）
df_post = (
    df[['post_id', 'platform_collapsed', 'reblogs_count'] + narrative_frames + indicators]
    .dropna()
    .groupby(['post_id', 'platform_collapsed'], as_index=False)
    .mean()
)

# 7) 拟合函数：frames + 指示 + frames×指示（Negative Binomial + HC3）
def fit_model_with_interactions(frame_df, platform_name, outcome='reblogs_count'):
    data_ = frame_df[frame_df['platform_collapsed'] == platform_name].copy()
    print(data_.dropna().shape, data_.shape)

    frame_terms = " + ".join(narrative_frames)
    main_inds   = " + ".join(indicators)
    inter_terms = " + ".join([f"{f}:{z}" for f in narrative_frames for z in indicators])

    formula = f"{outcome} ~ {frame_terms} + {main_inds} + {inter_terms}"
    print(formula)

    model = smf.glm(
        formula=formula,
        data=data_,
        family=sm.families.NegativeBinomial()
    ).fit(cov_type='HC3')

    od_ratio = model.pearson_chi2 / model.df_resid if model.df_resid > 0 else np.nan
    return model, {'overdispersion_ratio': od_ratio, 'n': len(data_)}

# 8) 分平台拟合
models = {}
diagnostics = {}
for plat in ['truth', 'Bsky/Mastodon']:
    m, diag = fit_model_with_interactions(df_post, plat, outcome='reblogs_count')
    models[plat] = m
    diagnostics[plat] = diag
    print(f"\n=== Model Summary for {plat} ===\n")
    print(m.summary())
    print("\n[Diagnostics]", diag)
    print("\n" + "="*80 + "\n")

# 9) 整理系数（适配 multi-hot 指示变量命名）
def tidy_glm(model, platform_name, narrative_frames, indicators):
    """
    返回列：
    term, coef, se, z, p, ci_low, ci_high, platform, term_type, frame, group, sig
    其中 group 取自 {BidenPos,BidenNeg,TrumpPos,TrumpNeg}
    """
    params = model.params
    bse    = model.bse
    pvals  = model.pvalues
    zvals  = params / bse
    ci     = model.conf_int()
    ci.columns = ["ci_low", "ci_high"]

    df_coef = (
        pd.DataFrame({
            "term": params.index,
            "coef": params.values,
            "se":   bse.values,
            "z":    zvals.values,
            "p":    pvals.values,
        }).join(ci, how="left")
    )
    df_coef["platform"] = platform_name

    # 解析项类型
    def parse_term(t):
        if t == "Intercept":
            return pd.Series({"term_type":"intercept", "frame":None, "group":None})
        if ":" in t:
            left, right = t.split(":", 1)
            frame = left if left in narrative_frames else None
            group = right if right in indicators else None
            return pd.Series({"term_type":"interaction", "frame":frame, "group":group})
        # main
        if t in narrative_frames:
            return pd.Series({"term_type":"main", "frame":t, "group":None})
        if t in indicators:
            return pd.Series({"term_type":"group_main", "frame":None, "group":t})
        # fallback
        return pd.Series({"term_type":"other", "frame":None, "group":None})

    parsed = df_coef["term"].apply(parse_term)
    df_coef = pd.concat([df_coef, parsed], axis=1)

    def star(p):
        if p < 0.001: return "***"
        if p < 0.01:  return "**"
        if p < 0.05:  return "*"
        if p < 0.10:  return "·"
        return ""
    df_coef["sig"] = df_coef["p"].apply(star)
    return df_coef

# 10) 生成 tidy 表 & 查看显著交互
tidy_list = []
for plat in ['truth', 'Bsky/Mastodon']:
    tidy_list.append(tidy_glm(models[plat], plat, narrative_frames, indicators))
coef_tidy = pd.concat(tidy_list, ignore_index=True)

interaction_sig = (
    coef_tidy
    .query("term_type == 'interaction'")
    .sort_values(["platform", "p"])
)

for plat in ['truth', 'Bsky/Mastodon']:
    print(f"\n=== Significant interactions @ {plat} (p < 0.05) ===")
    display(
        interaction_sig
        .query("platform == @plat and p < 0.05")
        .loc[:, ["term","frame","group","coef","se","z","p","ci_low","ci_high","sig"]]
        .head(30)
    )


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from typing import Dict, Tuple
# 你已有：models = {'truth': <GLMResults>, 'Bsky/Mastodon': <GLMResults>}
#          narrative_frames = 下划线命名的 frame 列名列表（不含 Intercept）

PLATFORM_LABEL = {'truth': 'TruthSocial', 'Bsky/Mastodon': 'Bluesky and Mastodon'}
GROUPS = ['Trump+','Trump-','Biden+','Biden-']

def _term_group_main(g: str) -> str:
    return f"C(target_attitude, Treatment(reference='Neutral'))[T.{g}]"

def _term_interaction(frame: str, g: str) -> str:
    return f"{frame}:C(target_attitude, Treatment(reference='Neutral'))[T.{g}]"

def lincom_ttest(model, terms):
    """
    对若干参数做线性组合检验：sum(terms) == 0 ？
    返回 (coef, se, z, p)；缺失的参数按 0 处理
    """
    idx = model.params.index
    L = np.zeros(len(idx))
    for t in terms:
        if t in idx:
            L[idx.get_loc(t)] += 1.0
        else:
            # 缺失项视为 0（例如模型里没出现的交互）
            L += 0.0
    res = model.t_test(L)  # Wald
    # 兼容不同返回形状
    coef = float(np.asarray(res.effect).squeeze())
    se   = float(np.asarray(res.sd).squeeze())
    z    = float(np.asarray(res.tvalue).squeeze())
    p    = float(np.asarray(res.pvalue).squeeze())
    return coef, se, z, p

def compute_combined_tables(models: Dict[str, object],
                            narrative_frames: list,
                            alpha: float = 0.05
                           ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    为每个平台构建一个大表（行 = frames + Intercept；列 = 平台×group），
    其中：
      - frame 行 = β_frame + β_(frame×group) 的 lincom 结果
      - Intercept 行 = β_0 + β_group (相对 Neutral 的总截距)
    """
    frames_plus_intercept = narrative_frames + ["Intercept"]

    all_coef, all_p, all_sig = [], [], []
    for plat_key, model in models.items():
        plat_label = PLATFORM_LABEL.get(plat_key, plat_key)

        cols = [f"{plat_label} {g}" for g in GROUPS]
        coef_df = pd.DataFrame(index=frames_plus_intercept, columns=cols, dtype=float)
        p_df    = pd.DataFrame(index=frames_plus_intercept, columns=cols, dtype=float)

        for g in GROUPS:
            col = f"{plat_label} {g}"

            # Intercept 行：β0 + β_group
            g_term = _term_group_main(g)
            coef_int, se_int, z_int, p_int = lincom_ttest(model, ["Intercept", g_term])
            coef_df.loc["Intercept", col] = coef_int
            p_df.loc["Intercept", col]    = p_int

            # Frame 行：β_frame + β_(frame×group)
            for f in narrative_frames:
                coef_f, se_f, z_f, p_f = lincom_ttest(model, [f, _term_interaction(f, g)])
                coef_df.loc[f, col] = coef_f
                p_df.loc[f, col]    = p_f

        sig_df = (p_df < alpha)
        all_coef.append(coef_df)
        all_p.append(p_df)
        all_sig.append(sig_df)

    coef_all = pd.concat(all_coef, axis=1)
    p_all    = pd.concat(all_p, axis=1)
    sig_all  = pd.concat(all_sig, axis=1)
    return coef_all, p_all, sig_all


def plot_gray_mask_heatmap(coef_all: pd.DataFrame,
                           sig_all: pd.DataFrame,
                           vmin: float = -1, vmax: float = 1,
                           title: str = "Relative effects (frame + frame×group); Intercept = group main"
                          ):
    """灰底 + 显著彩色叠加；所有 cell 均标注数值，非显著用灰字。"""
    plt.figure(figsize=(14, 9), dpi=300)

    # 灰底层
    sns.heatmap(coef_all, cmap="Greys", cbar=False,
                mask=sig_all,
                linewidths=0.5, linecolor='gray', vmin=-100, vmax=100)

    # 显著彩色层（mask=~sig）
    sns.heatmap(coef_all, cmap="coolwarm", center=0, vmin=vmin, vmax=vmax,
                mask=~sig_all,
                cbar_kws={"label": f"Coefficient (cap at [{vmin}, {vmax}])"},
                linewidths=0.5, linecolor='lightgray')

    # 数值注释：显著黑字，不显著灰字
    ax = plt.gca()
    annot = coef_all.applymap(lambda v: "" if pd.isna(v) else f"{v:.2f}")
    for i, r in enumerate(coef_all.index):
        for j, c in enumerate(coef_all.columns):
            txt = annot.loc[r, c]
            if not txt:
                continue
            color = 'white' if sig_all.loc[r, c] else 'white'
            ax.text(j+0.5, i+0.5, txt, ha='center', va='center', fontsize=12, color=color)
    # yticks remove "_"
    yticks_labels = ax.get_yticklabels()
    yticks_labels = [lbl.get_text().replace("_", " ") for lbl in yticks_labels]
    ax.set_yticklabels(yticks_labels, fontsize=12)

    plt.ylabel("Frame (base + interaction)  +  Intercept=row = group main", fontsize=12, fontweight='bold')
    plt.xticks(rotation=45, ha='right', fontsize=14)
    plt.yticks(rotation=0, fontsize=14)
    #plt.title(title, fontsize=13)
    plt.tight_layout()
    plt.show()


In [ ]:
# 1) 计算合成系数 + p + 显著性
coef_all, p_all, sig_all = compute_combined_tables(models, narrative_frames, alpha=0.05)

# 2) 画图（不显著灰掉）

plot_gray_mask_heatmap(coef_all, sig_all, vmin=-2, vmax=2)

# 3) 如果要导出便于复查
coef_all.to_csv("combined_coef_base_plus_interaction.csv")
p_all.to_csv("combined_coef_pvalues.csv")
sig_all.to_csv("combined_coef_significance_mask.csv")


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from typing import Dict, Tuple

# models = {'truth': <GLMResults>, 'Bsky/Mastodon': <GLMResults>}
# narrative_frames = 下划线命名的 frame 列名列表（不含 Intercept）

PLATFORM_LABEL = {'truth': 'TruthSocial', 'Bsky/Mastodon': 'Bluesky and Mastodon'}

# 展示用分组标签（列名里显示）
DISPLAY_GROUPS = ['Trump+','Trump-','Biden+','Biden-']
# 显示标签 -> 实际指示变量列名（方案B里我们建的列）
INDICATOR_FOR = {
    'Trump+': 'TrumpPos',
    'Trump-': 'TrumpNeg',
    'Biden+': 'BidenPos',
    'Biden-': 'BidenNeg',
}

def _term_group_main(ind_name: str) -> str:
    """主效应项（multi-hot 指示变量列名本身）"""
    return ind_name  # e.g., 'TrumpPos'

def _term_interaction(frame: str, ind_name: str) -> str:
    """交互项名：frame:Indicator"""
    return f"{frame}:{ind_name}"

def lincom_ttest(model, terms):
    """
    Wald 线性组合检验：sum(selected terms) == 0 ?
    返回 (coef, se, z, p)；缺失的参数按 0 处理
    """
    idx = model.params.index
    L = np.zeros(len(idx))
    for t in terms:
        if t in idx:
            L[idx.get_loc(t)] += 1.0
    res = model.t_test(L)
    coef = float(np.asarray(res.effect).squeeze())
    se   = float(np.asarray(res.sd).squeeze())
    z    = float(np.asarray(res.tvalue).squeeze())
    p    = float(np.asarray(res.pvalue).squeeze())
    return coef, se, z, p

def compute_combined_tables(models: Dict[str, object],
                            narrative_frames: list,
                            alpha: float = 0.05
                           ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    行 = frames + Intercept；列 = 平台 × {Trump+, Trump-, Biden+, Biden-}
      - Frame 行 = β_frame + β_(frame:Indicator)
      - Intercept 行 = β_0 + β_(Indicator)
    """
    frames_plus_intercept = narrative_frames + ["Intercept"]

    all_coef, all_p, all_sig = [], [], []
    for plat_key, model in models.items():
        plat_label = PLATFORM_LABEL.get(plat_key, plat_key)

        cols = [f"{plat_label} {gdisp}" for gdisp in DISPLAY_GROUPS]
        coef_df = pd.DataFrame(index=frames_plus_intercept, columns=cols, dtype=float)
        p_df    = pd.DataFrame(index=frames_plus_intercept, columns=cols, dtype=float)

        for gdisp in DISPLAY_GROUPS:
            ind = INDICATOR_FOR[gdisp]
            col = f"{plat_label} {gdisp}"

            # Intercept 行：β0 + β_(Indicator)
            coef_int, se_int, z_int, p_int = lincom_ttest(model, ["Intercept", _term_group_main(ind)])
            coef_df.loc["Intercept", col] = coef_int
            p_df.loc["Intercept", col]    = p_int

            # Frame 行：β_frame + β_(frame:Indicator)
            for f in narrative_frames:
                coef_f, se_f, z_f, p_f = lincom_ttest(model, [f, _term_interaction(f, ind)])
                coef_df.loc[f, col] = coef_f
                p_df.loc[f, col]    = p_f

        sig_df = (p_df < alpha)
        all_coef.append(coef_df)
        all_p.append(p_df)
        all_sig.append(sig_df)

    coef_all = pd.concat(all_coef, axis=1)
    p_all    = pd.concat(all_p, axis=1)
    sig_all  = pd.concat(all_sig, axis=1)
    return coef_all, p_all, sig_all

def plot_gray_mask_heatmap(coef_all: pd.DataFrame,
                           sig_all: pd.DataFrame,
                           vmin: float = -1, vmax: float = 1,
                           title: str = "Relative effects (frame + frame×indicator); Intercept = β0 + indicator"):
    """
    严格按你给的样式：
      1) 底层灰度 + 注释（所有格子）；
      2) 顶层仅显著彩色（mask=~sig）；非显著“显白”（靠顶层未覆盖 & 灰底很浅实现）。
    """
    # 注释文本
    annot = coef_all.applymap(lambda v: "" if pd.isna(v) else f"{v:.2f}")

    # 可选：若想让非显著更“白”，也可把一个复制版设成 NaN（这里按你的风格保留）
    masked_data = coef_all.copy()
    masked_data[~sig_all] = np.nan  # 若你用它作底层，即可真正把非显著涂白

    plt.figure(figsize=(14, 9), dpi=300)

    # 底层：灰度 + 注释（不加 mask）
    sns.heatmap(
        coef_all, cmap="Greys", cbar=False,
        annot=annot, fmt="",
        linewidths=0.5, linecolor='gray'
    )

    # 顶层：仅显著位置着色（mask=~sig_all）
    sns.heatmap(
        coef_all, cmap="coolwarm", center=0, vmin=vmin, vmax=vmax,
        mask=~sig_all, annot=False, fmt="",
        cbar_kws={"label": f"Coefficient (cap at [{vmin}, {vmax}])"},
        linewidths=0.5, linecolor='gray'
    )

    ax = plt.gca()
    # y 轴标签去下划线
    yticks = [lbl.get_text().replace("_", " ") for lbl in ax.get_yticklabels()]
    ax.set_yticklabels(yticks, fontsize=12)

    plt.ylabel("Frame (base + interaction)  ·  Intercept row = β0 + indicator", fontsize=12, fontweight='bold')
    plt.xticks(rotation=45, ha='right', fontsize=14)
    plt.yticks(rotation=0, fontsize=14)
    # plt.title(title, fontsize=13)
    plt.tight_layout()
    plt.show()


In [ ]:
coef_all, p_all, sig_all = compute_combined_tables(models, narrative_frames, alpha=0.05)
plot_gray_mask_heatmap(coef_all, sig_all, vmin=-1, vmax=1)


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
import re
import seaborn as sns
import matplotlib.pyplot as plt

# 1) 定义 frame 列（原始名 -> 下划线）
narrative_frames = [
    "Persecution", "Corruption", "Accountability", "Irony",
    "Heroism", "Civic Critique", "Moral Decay", "Media Manipulation",
    "Strategic Pragmatism", "Cultural Identity"
]

# 2) 拷贝 & 统一列名（只保留 root posts；每个 post 一行）
df = (
    narrative_analysis_df
    # .loc[narrative_analysis_df['in_reply_to_id'].isna(), :]
    .drop_duplicates(subset=['post_id'])
    .copy()
)
df.rename(columns={"Irony/Detachment": "Irony"}, inplace=True)
df.rename(columns={f: f.replace(" ", "_") for f in narrative_frames}, inplace=True)
narrative_frames = [f.replace(" ", "_") for f in narrative_frames]

# 3) multi-hot 指示变量（避免符号，用安全列名）
df['BidenPos'] = (df['biden']  == 'positive').astype(int)
df['BidenNeg'] = (df['biden']  == 'negative').astype(int)
df['TrumpPos'] = (df['trump']  == 'positive').astype(int)
df['TrumpNeg'] = (df['trump']  == 'negative').astype(int)
indicators = ['BidenPos','BidenNeg','TrumpPos','TrumpNeg']
DISPLAY_GROUPS = ['Trump+','Trump-','Biden+','Biden-']
INDICATOR_FOR  = {'Trump+':'TrumpPos','Trump-':'TrumpNeg','Biden+':'BidenPos','Biden-':'BidenNeg'}

# 4) 平台合并：Truth vs Bsky/Mastodon（一次即可）
platform_lower = df['platform'].str.lower()
df['platform_collapsed'] = np.where(
    platform_lower.isin(['bluesky', 'mastodon']),
    'Bsky/Mastodon',
    platform_lower
)
df = df[df['platform_collapsed'].isin(['truth', 'Bsky/Mastodon'])].copy()

# 5) （可选）标准化 frame 分数
scaler = StandardScaler()
df[narrative_frames] = scaler.fit_transform(df[narrative_frames])

# 6) 聚合到 post level（不再按 target 分组）
df_post = (
    df[['post_id', 'platform_collapsed', 'reblogs_count'] + narrative_frames + indicators]
    .dropna()
    .groupby(['post_id', 'platform_collapsed'], as_index=False)
    .mean()
)

# 7) 拟合函数：frames + 指示 + frames×指示（Negative Binomial + HC3）
def fit_model_with_interactions(frame_df, platform_name, outcome='reblogs_count'):
    data_ = frame_df[frame_df['platform_collapsed'] == platform_name].copy()
    frame_terms = " + ".join(narrative_frames)
    main_inds   = " + ".join(indicators)
    inter_terms = " + ".join([f"{f}:{z}" for f in narrative_frames for z in indicators])
    formula = f"{outcome} ~ {frame_terms} + {main_inds} + {inter_terms}"
    model = smf.glm(
        formula=formula,
        data=data_,
        family=sm.families.NegativeBinomial()
    ).fit(cov_type='HC3')
    od_ratio = model.pearson_chi2 / model.df_resid if model.df_resid > 0 else np.nan
    return model, {'overdispersion_ratio': od_ratio, 'n': len(data_)}

# 8) 分平台拟合
models = {}
diagnostics = {}
for plat in ['truth', 'Bsky/Mastodon']:
    m, diag = fit_model_with_interactions(df_post, plat, outcome='reblogs_count')
    models[plat] = m
    diagnostics[plat] = diag
    print(f"\n=== Model Summary for {plat} ===\n")
    print(m.summary())
    print("\n[Diagnostics]", diag)
    print("\n" + "="*80 + "\n")

# =========================
# 9) 交互 + baseline 可视化（两个 panel + 窄白分隔）
#    主体: β_(frame×indicator)
#    右侧: β_frame（Frame Baseline）
#    底部: 单行 β_group（vs Neutral），并在 baseline 列底部填 β0（Intercept）
# =========================
PLATFORM_LABEL = {'truth': 'TruthSocial', 'Bsky/Mastodon': 'Bluesky and Mastodon'}
FRAMES_ORDER   = narrative_frames[:]  # 行顺序只含 frames

def lincomb(model, terms):
    """返回线性组合/单项 (coef, pval)"""
    idx = model.params.index
    L = np.zeros(len(idx))
    for t in (terms if isinstance(terms, (list, tuple)) else [terms]):
        if t in idx:
            L[idx.get_loc(t)] += 1.0
    res = model.t_test(L)
    return float(np.asarray(res.effect).squeeze()), float(np.asarray(res.pvalue).squeeze())

def _term_interaction(frame: str, ind_name: str) -> str:
    return f"{frame}:{ind_name}"

def build_panel(model, plat_key):
    """
    单平台面板：
      - 主体列：各 group（交互项 β_(f×g)）
      - 右侧列：Baseline • Frame (= β_frame)
      - 输出：coef_df, pval_df, baseline_col_name, group_row (β_group), group_row_p, beta0, p0
    """
    plat_label = PLATFORM_LABEL.get(plat_key, plat_key)
    cols = [f"{plat_label} {g}" for g in DISPLAY_GROUPS]

    coef = pd.DataFrame(index=FRAMES_ORDER, columns=cols, dtype=float)
    pval = pd.DataFrame(index=FRAMES_ORDER, columns=cols, dtype=float)
    for gdisp in DISPLAY_GROUPS:
        ind = INDICATOR_FOR[gdisp]
        col = f"{plat_label} {gdisp}"
        for f in FRAMES_ORDER:
            c, p = lincomb(model, _term_interaction(f, ind))
            coef.loc[f, col] = c
            pval.loc[f, col] = p

    # 右侧 Fix Effects（β_frame）
    right_name = f"Intercept + Frame Main Effect ({plat_label})"
    coef[right_name] = [lincomb(model, f)[0] for f in FRAMES_ORDER]
    pval[right_name] = [lincomb(model, f)[1] for f in FRAMES_ORDER]

    # β_group（相对 Neutral）
    group_row   = pd.Series(index=list(coef.columns), dtype=float)
    group_row_p = pd.Series(index=list(pval.columns), dtype=float)
    for gdisp in DISPLAY_GROUPS:
        ind = INDICATOR_FOR[gdisp]
        c, p = lincomb(model, ind)  # 仅 group main effect
        group_row[f"{plat_label} {gdisp}"] = c
        group_row_p[f"{plat_label} {gdisp}"] = p
    group_row[right_name]   = np.nan
    group_row_p[right_name] = np.nan

    # 取 β0（universal intercept）
    beta0, p0 = lincomb(model, "Intercept")
    return coef, pval, right_name, group_row, group_row_p, beta0, p0

# 构造两个 panel
coef_t, p_t, right_t, group_t, group_p_t, beta0_t, p0_t = build_panel(models['truth'], 'truth')
coef_b, p_b, right_b, group_b, group_p_b, beta0_b, p0_b = build_panel(models['Bsky/Mastodon'], 'Bsky/Mastodon')

# 列拼接（Truth 在左，Bsky 在右）
coef_panels = pd.concat([coef_t, coef_b], axis=1)
pval_panels = pd.concat([p_t,  p_b], axis=1)

# 底部单行：β_group（vs Neutral），并把 β0 填到各自 panel 的 baseline 列底部格
row_name = "Group Main Effect (vs Neutral)"
all_cols = list(coef_panels.columns)
bottom_coef = pd.DataFrame(index=[row_name], columns=all_cols, dtype=float)
bottom_pval = pd.DataFrame(index=[row_name], columns=all_cols, dtype=float)

bottom_coef.loc[row_name, group_t.index]   = group_t.values
bottom_pval.loc[row_name, group_p_t.index] = group_p_t.values
bottom_coef.loc[row_name, group_b.index]   = group_b.values
bottom_pval.loc[row_name, group_p_b.index] = group_p_b.values

# β0 → baseline 列底部
bottom_coef.loc[row_name, right_t] = beta0_t
bottom_pval.loc[row_name, right_t] = p0_t
bottom_coef.loc[row_name, right_b] = beta0_b
bottom_pval.loc[row_name, right_b] = p0_b

# 最终绘图数据
coef_plot = pd.concat([coef_panels.loc[FRAMES_ORDER], bottom_coef], axis=0)
pval_plot = pd.concat([pval_panels.loc[FRAMES_ORDER], bottom_pval], axis=0)

# 10) 可视化：两个 panel + 窄白条分隔
alpha = 0.05
signif_mask = (pval_plot < alpha)
annot = coef_plot.applymap(lambda v: "" if pd.isna(v) else f"{v:.2f}")

plt.figure(figsize=(16, 10), dpi=300)

# 灰底
ax = sns.heatmap(
    coef_plot, cmap="Greys", cbar=False,
    annot=annot, fmt="", linewidths=0.4, linecolor='lightgray',
    annot_kws={"color": "white", "size":12},
)
# 显著叠色
sns.heatmap(
    coef_plot, cmap="coolwarm", center=0,
    mask=~signif_mask, annot=False,
    cbar_kws={"label": "Coefficient (significant only)"},
    linewidths=0.4, linecolor='lightgray',
    vmin = -0.5, vmax=0.5
)

# 窄白分隔线：panel 分界 & baseline 列左侧 & frames/底部之间
cols = list(coef_plot.columns)
x_truth_baseline = cols.index(right_t)           # Truth baseline 列左边界
x_bsky_baseline  = cols.index(right_b)           # Bsky  baseline 列左边界
x_panel_split    = cols.index(right_t) + 1       # 两个 panel 分界（在 Truth baseline 之后）
y_baseline       = len(FRAMES_ORDER)             # frames 与底部单行之间

for x in [x_truth_baseline, x_panel_split, x_bsky_baseline]:
    if x  is x_panel_split:
        ax.vlines(x, *ax.get_ylim(), colors="white", linewidth=12.0, zorder=10)
    else:
        ax.vlines(x, *ax.get_ylim(), colors="white", linewidth=8.0, zorder=10)
ax.hlines(y_baseline, *ax.get_xlim(), colors="white", linewidth=6.0, zorder=10)

# 标签和 panel 标题
ax.set_yticklabels([lbl.get_text().replace("_", " ") for lbl in ax.get_yticklabels()],fontweight='bold')
plt.xticks(rotation=45, ha='right', fontweight='bold')

def panel_center_x(panel_cols):
    xs = [cols.index(c) + 0.5 for c in panel_cols]
    return (min(xs) + max(xs)) / 2

truth_panel_cols = [c for c in cols if c.startswith(PLATFORM_LABEL['truth'])] + [right_t]

bsky_panel_cols  = [c for c in cols if c.startswith(PLATFORM_LABEL['Bsky/Mastodon'])] + [right_b]

ax.text(panel_center_x(truth_panel_cols), -0.6, PLATFORM_LABEL['truth'],
        ha='center', va='bottom', fontsize=14, fontweight='bold', transform=ax.transData)
ax.text(panel_center_x(bsky_panel_cols), -0.6, PLATFORM_LABEL['Bsky/Mastodon'],
        ha='center', va='bottom', fontsize=14, fontweight='bold', transform=ax.transData)

plt.ylabel("Frames  +  Group Effect (vs Neutral)", fontsize=12, fontweight='bold')
plt.xlabel("Targets per panel  +  Fix Effects", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()





In [ ]:
from scipy import stats

def _finite_clean(df, cols):
    """仅保留这些列全为有限值的行"""
    return df.loc[np.isfinite(df[cols]).all(axis=1)].copy()

def _present_inds(df, inds):
    """保留在该子样本中出现过(求和>0)的指示变量"""
    present = [z for z in inds if df[z].sum() > 0]
    removed = [z for z in inds if z not in present]
    return present, removed

def _fit_nb_glm(data, formula):
    """先用 Poisson 得到起点，再拟合 NB (alpha=1.0, log link)，返回 GLMResults"""
    # Poisson 起点
    pois = smf.glm(formula=formula, data=data, family=sm.families.Poisson()).fit()
    nb_mod = smf.glm(
        formula=formula,
        data=data,
        family=sm.families.NegativeBinomial(alpha=1.0, link=sm.families.links.log())
    )
    start = pd.Series(0.0, index=nb_mod.exog_names)
    for k, v in pois.params.items():
        if k in start.index:
            start[k] = v
    res = nb_mod.fit(start_params=start.values)  # 用常规协方差以便 LRT/AIC/BIC
    return res

def _model_metrics(res, llf_null):
    n = int(res.nobs)
    k = len(res.params)
    aic = res.aic
    bic = -2.0 * res.llf + k * np.log(n)  # statsmodels 的 GLM 有时不直接给 bic
    r2  = 1.0 - (res.llf / llf_null)
    return dict(n=n, k=k, llf=float(res.llf), AIC=float(aic), BIC=float(bic), R2=float(r2))

def compare_nb_interactions(df_post, platform_name, outcome='reblogs_count',
                            frames=None, inds=None):
    if frames is None:
        frames = narrative_frames
    if inds is None:
        inds = indicators

    data_ = df_post[df_post['platform_collapsed'] == platform_name].copy()
    # 清洗
    use_cols = [outcome] + frames + inds
    data_ = _finite_clean(data_, use_cols)
    data_[outcome] = data_[outcome].clip(lower=0)
    # 仅保留出现过的指示变量
    present, removed = _present_inds(data_, inds)

    # 公式拼接
    frame_terms = " + ".join(frames)
    formula_no  = f"{outcome} ~ {frame_terms}" + (f" + {' + '.join(present)}" if present else "")
    inter_terms = " + ".join([f"{f}:{z}" for f in frames for z in present]) if present else ""
    formula_int = formula_no + (f" + {inter_terms}" if inter_terms else "")

    # Null 模型（仅截距），用于 R2
    null_res = _fit_nb_glm(data_, f"{outcome} ~ 1")
    llf_null = null_res.llf

    # 拟合：无交互 vs 有交互
    m_no  = _fit_nb_glm(data_, formula_no)
    m_int = _fit_nb_glm(data_, formula_int)

    # 信息准则
    met_no  = _model_metrics(m_no,  llf_null)
    met_int = _model_metrics(m_int, llf_null)

    # 似然比检验（嵌套：有交互 vs 无交互）
    lr_stat = 2.0 * (m_int.llf - m_no.llf)
    df_diff = len(m_int.params) - len(m_no.params)
    p_val   = stats.chi2.sf(lr_stat, df_diff) if df_diff > 0 else np.nan

    result = {
        'platform': platform_name,
        'present_inds': present,
        'removed_inds': removed,
        'formula_no': formula_no,
        'formula_int': formula_int,
        'AIC_no': met_no['AIC'],
        'AIC_int': met_int['AIC'],
        'ΔAIC': met_int['AIC'] - met_no['AIC'],
        'BIC_no': met_no['BIC'],
        'BIC_int': met_int['BIC'],
        'ΔBIC': met_int['BIC'] - met_no['BIC'],
        'LRT_χ2': float(lr_stat),
        'LRT_df': int(df_diff),
        'LRT_p': float(p_val),
        'R2_no': met_no['R2'],
        'R2_int': met_int['R2'],
        'ΔR2': met_int['R2'] - met_no['R2'],
        'n': met_no['n']
    }
    return result, m_no, m_int

# === 跑两个平台 & 汇总表 ===
rows = []
store = {}
for plat in ['truth', 'Bsky/Mastodon']:
    res, m_no, m_int = compare_nb_interactions(df_post, plat, outcome='reblogs_count',
                                               frames=narrative_frames, inds=indicators)
    rows.append(res)
    store[plat] = {'no_int': m_no, 'with_int': m_int, 'meta': res}

summary = pd.DataFrame(rows)[[
    'platform', 'n',
    'AIC_no', 'AIC_int', 'ΔAIC',
    'BIC_no', 'BIC_int', 'ΔBIC',
    'LRT_χ2', 'LRT_df', 'LRT_p',
    'R2_no', 'R2_int', 'ΔR2'
]].copy()

# 友好显示：平台标签 & 排序
PLATFORM_LABEL = {'truth': 'TruthSocial', 'Bsky/Mastodon': 'Bluesky and Mastodon'}
summary['platform'] = summary['platform'].map(lambda x: PLATFORM_LABEL.get(x, x))
summary = summary.set_index('platform')

print("\n=== NB 模型：加入交互项 前 vs 后 的对比（越低越好：AIC/BIC；越高越好：R2） ===")
display(summary.style.format({
    'AIC_no':'{:.1f}','AIC_int':'{:.1f}','ΔAIC':'{:+.1f}',
    'BIC_no':'{:.1f}','BIC_int':'{:.1f}','ΔBIC':'{:+.1f}',
    'LRT_χ2':'{:.2f}','LRT_p':'{:.3g}',
    'R2_no':'{:.4f}','R2_int':'{:.4f}','ΔR2':'{:+.4f}',
    'n':'{:.0f}'
}).background_gradient(subset=['ΔAIC','ΔBIC','ΔR2'], cmap='RdYlGn_r'))


In [ ]:
# ============================== 全部代码 ==============================
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt

# ============ 0) 基础配置 ============
narrative_frames = [
    "Persecution", "Corruption", "Accountability", "Irony",
    "Heroism", "Civic Critique", "Moral Decay", "Media Manipulation",
    "Strategic Pragmatism", "Cultural Identity"
]
narrative_analysis_df = narrative_analysis_df.drop_duplicates(subset=['post_id']a)
# ============ 1) 生成 reinforce_rate & 合并到主表 ============
NDIR_df['reinforce_rate'] = NDIR_df['classification'].apply(lambda x: 1 if x == 'reinforce' else 0)
NDIR_df_groupby = NDIR_df.groupby('root_post_id').agg({'reinforce_rate': 'mean'}).reset_index()

narrative_analysis_df.post_id = narrative_analysis_df.post_id.astype(str)
NDIR_df_groupby.root_post_id  = NDIR_df_groupby.root_post_id.astype(str)

df = narrative_analysis_df.merge(
    NDIR_df_groupby.rename(columns={'root_post_id': 'post_id'}),
    on='post_id', how='right'
)

# 统一列名（Irony/Detachment -> Irony；空格 -> 下划线）
df.rename(columns={"Irony/Detachment": "Irony"}, inplace=True)
df.rename(columns={f: f.replace(" ", "_") for f in narrative_frames}, inplace=True)
narrative_frames = [f.replace(" ", "_") for f in narrative_frames]

# 仅保留有分数与reinforce的行
df.dropna(subset=narrative_frames + ['reinforce_rate'], inplace=True)

# ============ 2) 目标标签：允许多标签，explode ============
def get_target_attitudes(row):
    tags = []
    if row.get('biden') == 'positive':
        tags.append("Biden+")
    elif row.get('biden') == 'negative':
        tags.append("Biden-")
    if row.get('trump') == 'positive':
        tags.append("Trump+")
    elif row.get('trump') == 'negative':
        tags.append("Trump-")
    return tags if tags else ['Neutral']

df['target_attitudes'] = df.apply(get_target_attitudes, axis=1)
df = df.explode('target_attitudes').rename(columns={'target_attitudes': 'target_attitude'})

# ============ 3) 平台合并：Truth vs Bsky/Mastodon ============
df['platform_collapsed'] = np.where(
    df['platform'].str.lower().isin(['bluesky', 'mastodon']),
    'Bsky/Mastodon',
    df['platform'].str.lower()
)
df = df[df['platform_collapsed'].isin(['truth', 'Bsky/Mastodon'])].copy()

# ============ 4) 标准化 frame 分数 ============
scaler = StandardScaler()
df[narrative_frames] = scaler.fit_transform(df[narrative_frames])

# ============ 5) 防止 0/1 极端 ============
epsilon = 1e-4
df['reinforce_rate'] = df['reinforce_rate'].clip(epsilon, 1 - epsilon)

# ============ 6) 聚合到 post level ============
df_post = (
    df[['post_id', 'platform_collapsed', 'target_attitude', 'reinforce_rate'] + narrative_frames]
    .dropna()
    .groupby(['post_id', 'platform_collapsed', 'target_attitude'], as_index=False)
    .mean()
)

# ============ 7) 拟合：每个平台一个 Logit-GLM（含交互），稳健协方差 HC3 ============
def fit_model_with_interactions(frame_df, platform_name, outcome='reinforce_rate'):
    data_ = frame_df[frame_df['platform_collapsed'] == platform_name].copy()
    frame_terms = " + ".join(narrative_frames)
    interaction_terms = " + ".join([f"{f}:C(target_attitude, Treatment(reference='Neutral'))" for f in narrative_frames])
    formula = f"{outcome} ~ C(target_attitude, Treatment(reference='Neutral')) + {frame_terms} + {interaction_terms}"
    model = smf.glm(formula=formula, data=data_, family=sm.families.Binomial()).fit(cov_type='HC3')
    return model

models = {
    'truth': fit_model_with_interactions(df_post, 'truth', outcome='reinforce_rate'),
    'Bsky/Mastodon': fit_model_with_interactions(df_post, 'Bsky/Mastodon', outcome='reinforce_rate')
}

# ============ 8) 组装矩阵：只显示交互 + 两条 baseline（右侧 frame + 底部单行“group effect”） ============
PLATFORM_LABEL = {'truth': 'TruthSocial', 'Bsky/Mastodon': 'Bluesky and Mastodon'}
GROUPS = ['Trump+','Trump-','Biden+','Biden-']
FRAMES_ORDER = narrative_frames[:]  # 行顺序只含 frames

def _term_group_main(g: str) -> str:
    return f"C(target_attitude, Treatment(reference='Neutral'))[T.{g}]"

def _term_interaction(frame: str, g: str) -> str:
    return f"{frame}:C(target_attitude, Treatment(reference='Neutral'))[T.{g}]"

def lincomb(model, terms):
    """线性组合（或单项）系数与 p 值"""
    idx = model.params.index
    L = np.zeros(len(idx))
    for t in (terms if isinstance(terms, (list, tuple)) else [terms]):
        if t in idx:
            L[idx.get_loc(t)] += 1.0
    res = model.t_test(L)
    return float(np.asarray(res.effect).squeeze()), float(np.asarray(res.pvalue).squeeze())

def build_panel(model, plat_key):
    """
    构造单平台：
      - 主体：交互项 β_(frame×group)
      - 右侧：frame-baseline（β_frame）
      - 底部：仅 group 主效应（β_group，相对 Neutral）
    """
    plat_label = PLATFORM_LABEL.get(plat_key, plat_key)
    cols = [f"{plat_label} {g}" for g in GROUPS]

    coef = pd.DataFrame(index=FRAMES_ORDER, columns=cols, dtype=float)
    pval = pd.DataFrame(index=FRAMES_ORDER, columns=cols, dtype=float)
    for g in GROUPS:
        for f in FRAMES_ORDER:
            c, p = lincomb(model, _term_interaction(f, g))
            coef.loc[f, f"{plat_label} {g}"] = c
            pval.loc[f, f"{plat_label} {g}"] = p

    # 右侧 baseline（β_frame）
    right_name = f"Baseline • Frame ({plat_label})"
    coef[right_name] = [lincomb(model, f)[0] for f in FRAMES_ORDER]
    pval[right_name] = [lincomb(model, f)[1] for f in FRAMES_ORDER]

    # 底部：仅 group effect（β_group，相对 Neutral）
    bottom_name = f"Group • Intercept ({plat_label})"
    bottom_coef = pd.Series(index=list(coef.columns), dtype=float)
    bottom_pval = pd.Series(index=list(pval.columns), dtype=float)
    for g in GROUPS:
        c, p = lincomb(model, _term_group_main(g))   # 关键：只取 β_group
        bottom_coef[f"{plat_label} {g}"] = c
        bottom_pval[f"{plat_label} {g}"] = p
    bottom_coef[right_name] = np.nan  # 右下角留空
    bottom_pval[right_name] = np.nan

    bottom_coef = bottom_coef.to_frame().T
    bottom_coef.index = [bottom_name]
    bottom_pval = bottom_pval.to_frame().T
    bottom_pval.index = [bottom_name]
    return coef, pval, right_name, bottom_coef, bottom_pval

# 两个平台面板
coef_t, p_t, right_t, bottom_coef_t, bottom_p_t = build_panel(models['truth'], 'truth')
coef_b, p_b, right_b, bottom_coef_b, bottom_p_b = build_panel(models['Bsky/Mastodon'], 'Bsky/Mastodon')

# 列方向拼接（Truth 在左，Bsky 在右）
coef_panels = pd.concat([coef_t, coef_b], axis=1)
pval_panels = pd.concat([p_t,  p_b], axis=1)

# —— 底部“单行”：Baseline • Group Effect（把两平台的 group 主效应并到同一行）——
all_cols = coef_panels.columns
bottom_coef = pd.DataFrame(index=["Baseline • Group Effect (vs Neutral)"], columns=all_cols, dtype=float)
bottom_pval = pd.DataFrame(index=["Baseline • Group Effect (vs Neutral)"], columns=all_cols, dtype=float)
bottom_coef.loc["Baseline • Group Effect (vs Neutral)", bottom_coef_t.columns] = bottom_coef_t.iloc[0].values
bottom_pval.loc["Baseline • Group Effect (vs Neutral)", bottom_p_t.columns]    = bottom_p_t.iloc[0].values
bottom_coef.loc["Baseline • Group Effect (vs Neutral)", bottom_coef_b.columns] = bottom_coef_b.iloc[0].values
bottom_pval.loc["Baseline • Group Effect (vs Neutral)", bottom_p_b.columns]    = bottom_p_b.iloc[0].values

# 最终用于绘图的矩阵（行：frames + 单行 group-effect baseline）
coef_plot = pd.concat([coef_panels.loc[FRAMES_ORDER], bottom_coef], axis=0)
pval_plot = pd.concat([pval_panels.loc[FRAMES_ORDER], bottom_pval], axis=0)

# ============ 9) 可视化：两个 panel + 窄白条分隔 ============
alpha = 0.05
signif_mask = (pval_plot < alpha)
annot = coef_plot.applymap(lambda v: "" if pd.isna(v) else f"{v:.2f}")

plt.figure(figsize=(16, 10), dpi=300)

# 灰底
ax = sns.heatmap(
    coef_plot, cmap="Greys", cbar=False,
    annot=annot, fmt="", linewidths=0.4, linecolor='lightgray'
)
# 显著叠色
sns.heatmap(
    coef_plot, cmap="coolwarm", center=0, vmin=-1, vmax=1,
    mask=~signif_mask, annot=False,
    cbar_kws={"label": "Logit Coefficient (significant only)"},
    linewidths=0.4, linecolor='lightgray'
)

# —— 窄白分隔线（不占格宽）——
cols = list(coef_plot.columns)
# baseline frame 列左侧边界
x_truth_baseline = cols.index(right_t)
x_bsky_baseline  = cols.index(right_b)
# 两个 panel 的分界：在 Truth 的 baseline 列之后
x_panel_split    = cols.index(right_t) + 1
# frames 与底部 baseline（group effect）之间
y_baseline = len(FRAMES_ORDER)

for x in [x_truth_baseline, x_panel_split, x_bsky_baseline]:
    ax.vlines(x, *ax.get_ylim(), colors="white", linewidth=3.0, zorder=10)
ax.hlines(y_baseline, *ax.get_xlim(), colors="white", linewidth=3.0, zorder=10)

# 友好标签 & panel 顶部标题
ax.set_yticklabels([lbl.get_text().replace("_", " ") for lbl in ax.get_yticklabels()])
plt.xticks(rotation=45, ha='right')

def panel_center_x(panel_cols):
    xs = [cols.index(c) + 0.5 for c in panel_cols]
    return (min(xs) + max(xs)) / 2

truth_panel_cols = [c for c in cols if c.startswith(PLATFORM_LABEL['truth'])] + [right_t]
bsky_panel_cols  = [c for c in cols if c.startswith(PLATFORM_LABEL['Bsky/Mastodon'])] + [right_b]

ax.text(panel_center_x(truth_panel_cols), -0.6, PLATFORM_LABEL['truth'],
        ha='center', va='bottom', fontsize=14, fontweight='bold', transform=ax.transData)
ax.text(panel_center_x(bsky_panel_cols), -0.6, PLATFORM_LABEL['Bsky/Mastodon'],
        ha='center', va='bottom', fontsize=14, fontweight='bold', transform=ax.transData)

plt.ylabel("Frames  +  bottom: Baseline • Group Effect (vs Neutral)", fontsize=12, fontweight='bold')
plt.xlabel("Targets per panel  +  right: Baseline • Frame (per panel)", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
# ============================ 代码结束 ============================




In [ ]:
from scipy import stats

def fit_model_no_interactions(frame_df, platform_name, outcome='reinforce_rate'):
    data_ = frame_df[frame_df['platform_collapsed'] == platform_name].copy()
    frame_terms = " + ".join(narrative_frames)
    main_inds   = " + ".join(indicators)
    formula = f"{outcome} ~ {frame_terms} + {main_inds}"
    model = smf.glm(formula=formula, data=data_, family=sm.families.Binomial()).fit(cov_type='HC3')
    return model

def compare_models(frame_df, platform_name):
    m_no  = fit_model_no_interactions(frame_df, platform_name)
    m_int = fit_model_with_interactions(frame_df, platform_name)

    # 似然比检验
    lr_stat = 2 * (m_int.llf - m_no.llf)
    df_diff = m_int.df_model - m_no.df_model
    p_val   = stats.chi2.sf(lr_stat, df_diff)

    print(f"=== {platform_name} ===")
    print(f"No interactions:   AIC={m_no.aic:.1f}, BIC={m_no.bic:.1f}")
    print(f"With interactions: AIC={m_int.aic:.1f}, BIC={m_int.bic:.1f}")
    print(f"Likelihood Ratio Test: χ²({df_diff}) = {lr_stat:.2f}, p = {p_val:.4g}")
    print()

# 跑两平台
compare_models(df_post, 'truth')
compare_models(df_post, 'Bsky/Mastodon')


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt
from typing import Dict, Tuple

# ============ 0) 基础配置 ============
# Define narrative frame names (原始名 -> 下划线)
narrative_frames = [
    "Persecution", "Corruption", "Accountability", "Irony",
    "Heroism", "Civic Critique", "Moral Decay", "Media Manipulation",
    "Strategic Pragmatism", "Cultural Identity"
]

# ============ 1) 生成 reinforce_rate & 合并到主表 ============
NDIR_df['reinforce_rate'] = NDIR_df['classification'].apply(lambda x: 1 if x == 'reinforce' else 0)
NDIR_df_groupby = NDIR_df.groupby('root_post_id').agg({'reinforce_rate': 'mean'}).reset_index()

narrative_analysis_df.post_id = narrative_analysis_df.post_id.astype(str)
NDIR_df_groupby.root_post_id = NDIR_df_groupby.root_post_id.astype(str)

df = narrative_analysis_df.merge(
    NDIR_df_groupby.rename(columns={'root_post_id': 'post_id'}),
    on='post_id', how='right'
)

# 统一列名（Irony/Detachment -> Irony；空格 -> 下划线）
df.rename(columns={"Irony/Detachment": "Irony"}, inplace=True)
df.rename(columns={f: f.replace(" ", "_") for f in narrative_frames}, inplace=True)
narrative_frames = [f.replace(" ", "_") for f in narrative_frames]

# 仅保留有分数与reinforce的行
df.dropna(subset=narrative_frames + ['reinforce_rate'], inplace=True)

# ============ 2) multi-hot 指示变量（不再 explode） ============
df['BidenPos'] = (df['biden']  == 'positive').astype(int)
df['BidenNeg'] = (df['biden']  == 'negative').astype(int)
df['TrumpPos'] = (df['trump']  == 'positive').astype(int)
df['TrumpNeg'] = (df['trump']  == 'negative').astype(int)
indicators = ['BidenPos','BidenNeg','TrumpPos','TrumpNeg']

# ============ 3) 平台合并：Truth vs Bsky/Mastodon ============
platform_lower = df['platform'].str.lower()
df['platform_collapsed'] = np.where(
    platform_lower.isin(['bluesky', 'mastodon']),
    'Bsky/Mastodon',
    platform_lower
)
# 只保留两类平台
df = df[df['platform_collapsed'].isin(['truth', 'Bsky/Mastodon'])].copy()

# ============ 4) 标准化 frame 分数 ============
scaler = StandardScaler()
df[narrative_frames] = scaler.fit_transform(df[narrative_frames])

# ============ 5) 防止 0/1 极端 ============
epsilon = 1e-4
df['reinforce_rate'] = df['reinforce_rate'].clip(epsilon, 1 - epsilon)

# ============ 6) 聚合到 post level（不再包含 target_attitude） ============
df_post = (
    df[['post_id', 'platform_collapsed', 'reinforce_rate'] + narrative_frames + indicators]
    .dropna()
    .groupby(['post_id', 'platform_collapsed'], as_index=False)
    .mean()
)

# ============ 7) 拟合：每个平台一个 Logit-GLM（frames + 指示 + 交互），稳健协方差 HC3 ============
def fit_model_with_interactions(frame_df, platform_name, outcome='reinforce_rate'):
    data_ = frame_df[frame_df['platform_collapsed'] == platform_name].copy()

    frame_terms = " + ".join(narrative_frames)
    main_inds   = " + ".join(indicators)
    inter_terms = " + ".join([f"{f}:{z}" for f in narrative_frames for z in indicators])

    formula = f"{outcome} ~ {frame_terms} + {main_inds} + {inter_terms}"

    model = smf.glm(
        formula=formula,
        data=data_,
        family=sm.families.Binomial()
    ).fit(cov_type='HC3')
    return model

models = {
    'truth': fit_model_with_interactions(df_post, 'truth', outcome='reinforce_rate'),
    'Bsky/Mastodon': fit_model_with_interactions(df_post, 'Bsky/Mastodon', outcome='reinforce_rate')
}

# ============ 8) 线性组合检验（严格） ============
PLATFORM_LABEL = {'truth': 'TruthSocial', 'Bsky/Mastodon': 'Bluesky and Mastodon'}
DISPLAY_GROUPS = ['Trump+','Trump-','Biden+','Biden-']
INDICATOR_FOR = {'Trump+':'TrumpPos','Trump-':'TrumpNeg','Biden+':'BidenPos','Biden-':'BidenNeg'}

FRAMES_ORDER = narrative_frames[:]                      # 只含frame
FRAMES_PLUS_INTERCEPT = FRAMES_ORDER + ["Intercept"]    # 画图行顺序

def _term_group_main(ind_name: str) -> str:
    return ind_name  # 例如 'TrumpPos'

def _term_interaction(frame: str, ind_name: str) -> str:
    return f"{frame}:{ind_name}"

def lincom_ttest(model, terms):
    idx = model.params.index
    L = np.zeros(len(idx))
    for t in terms:
        if t in idx:
            L[idx.get_loc(t)] += 1.0
    res = model.t_test(L)  # Wald
    coef = float(np.asarray(res.effect).squeeze())
    se   = float(np.asarray(res.sd).squeeze())
    z    = float(np.asarray(res.tvalue).squeeze())
    p    = float(np.asarray(res.pvalue).squeeze())
    return coef, se, z, p

def compute_combined_tables(models: Dict[str, object],
                            alpha: float = 0.05
                           ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    all_coef, all_p, all_sig = [], [], []
    for plat_key, model in models.items():
        plat_label = PLATFORM_LABEL.get(plat_key, plat_key)
        cols = [f"{plat_label} {g}" for g in DISPLAY_GROUPS]
        coef_df = pd.DataFrame(index=FRAMES_PLUS_INTERCEPT, columns=cols, dtype=float)
        p_df    = pd.DataFrame(index=FRAMES_PLUS_INTERCEPT, columns=cols, dtype=float)

        for gdisp in DISPLAY_GROUPS:
            ind = INDICATOR_FOR[gdisp]
            col = f"{plat_label} {gdisp}"

            # Intercept 行：β0 + β_(Indicator)
            c_int, se_int, z_int, p_int = lincom_ttest(model, ["Intercept", _term_group_main(ind)])
            coef_df.loc["Intercept", col] = c_int
            p_df.loc["Intercept", col]    = p_int

            # Frame 行：β_frame + β_(frame:Indicator)
            for f in FRAMES_ORDER:
                c_f, se_f, z_f, p_f = lincom_ttest(model, [f, _term_interaction(f, ind)])
                coef_df.loc[f, col] = c_f
                p_df.loc[f, col]    = p_f

        sig_df = (p_df < alpha)
        all_coef.append(coef_df)
        all_p.append(p_df)
        all_sig.append(sig_df)

    coef_all = pd.concat(all_coef, axis=1)
    p_all    = pd.concat(all_p, axis=1)
    sig_all  = pd.concat(all_sig, axis=1)
    return coef_all, p_all, sig_all

coef_all, p_all, sig_all = compute_combined_tables(models, alpha=0.05)

# ============ 9) 画图：灰底 + 显著彩色覆盖；非显著白色 ============
row_order = FRAMES_PLUS_INTERCEPT
col_order = list(coef_all.columns)

reinforce_df = coef_all.loc[row_order, col_order].astype(float)
significance_mask = sig_all.loc[row_order, col_order].astype(bool)
reinforce_annot = reinforce_df.applymap(lambda v: "" if pd.isna(v) else f"{v:.2f}")

# 非显著置 NaN（若希望底层就“发白”可用 masked_data 作为底图）
masked_data = reinforce_df.copy()
masked_data[~significance_mask] = np.nan

plt.figure(figsize=(14, 9), dpi=300)

# 底层：灰度 + 注释
sns.heatmap(
    reinforce_df, cmap="Greys", cbar=False,
    annot=reinforce_annot, fmt="",
    linewidths=0.5, linecolor='gray'
)

# 顶层：仅显著位置着色
sns.heatmap(
    reinforce_df, cmap="coolwarm", center=0, vmin=-0.5, vmax=0.5,
    mask=~significance_mask, annot=False, fmt="",
    cbar_kws={"label": "Logit Coefficient (only significant cells colored)"},
    linewidths=0.5, linecolor='gray'
)

# y 轴标签去下划线
ax = plt.gca()
yticks_labels = [lbl.get_text().replace("_", " ") for lbl in ax.get_yticklabels()]
ax.set_yticklabels(yticks_labels)

plt.ylabel("Narrative Frame (plus Intercept)", fontsize=18, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=18)
plt.yticks(rotation=0, fontsize=18)
plt.tight_layout()
plt.show()



In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt
from typing import Dict, Tuple

# ============ 0) 基础配置 ============
narrative_frames = [
    "Persecution", "Corruption", "Accountability", "Irony",
    "Heroism", "Civic Critique", "Moral Decay", "Media Manipulation",
    "Strategic Pragmatism", "Cultural Identity"
]

# ============ 1) 生成 reinforce_rate & 合并到主表 ============
NDIR_df['reinforce_rate'] = NDIR_df['classification'].apply(lambda x: 1 if x == 'reinforce' else 0)
NDIR_df_groupby = NDIR_df.groupby('root_post_id').agg({'reinforce_rate': 'mean'}).reset_index()

narrative_analysis_df.post_id = narrative_analysis_df.post_id.astype(str)
NDIR_df_groupby.root_post_id = NDIR_df_groupby.root_post_id.astype(str)

df = narrative_analysis_df.merge(
    NDIR_df_groupby.rename(columns={'root_post_id': 'post_id'}),
    on='post_id', how='right'
)

# 统一列名（Irony/Detachment -> Irony；空格 -> 下划线）
df.rename(columns={"Irony/Detachment": "Irony"}, inplace=True)
df.rename(columns={f: f.replace(" ", "_") for f in narrative_frames}, inplace=True)
narrative_frames = [f.replace(" ", "_") for f in narrative_frames]

# 仅保留有分数与reinforce的行
df.dropna(subset=narrative_frames + ['reinforce_rate'], inplace=True)

# ============ 2) multi-hot 指示变量（不再 explode） ============
df['BidenPos'] = (df['biden']  == 'positive').astype(int)
df['BidenNeg'] = (df['biden']  == 'negative').astype(int)
df['TrumpPos'] = (df['trump']  == 'positive').astype(int)
df['TrumpNeg'] = (df['trump']  == 'negative').astype(int)
indicators = ['BidenPos','BidenNeg','TrumpPos','TrumpNeg']

# ============ 3) 平台合并：Truth vs Bsky/Mastodon ============
platform_lower = df['platform'].str.lower()
df['platform_collapsed'] = np.where(
    platform_lower.isin(['bluesky', 'mastodon']),
    'Bsky/Mastodon',
    platform_lower
)
df = df[df['platform_collapsed'].isin(['truth', 'Bsky/Mastodon'])].copy()

# ============ 4) 标准化 frame 分数 ============
scaler = StandardScaler()
df[narrative_frames] = scaler.fit_transform(df[narrative_frames])

# ============ 5) 防止 0/1 极端 ============
epsilon = 1e-4
df['reinforce_rate'] = df['reinforce_rate'].clip(epsilon, 1 - epsilon)

# ============ 6) 聚合到 post level ============
df_post = (
    df[['post_id', 'platform_collapsed', 'reinforce_rate'] + narrative_frames + indicators]
    .dropna()
    .groupby(['post_id', 'platform_collapsed'], as_index=False)
    .mean()
)

# ============ 7) 拟合：每个平台一个 Logit-GLM（frames + 指示 + 交互），HC3 ============
def fit_model_with_interactions(frame_df, platform_name, outcome='reinforce_rate'):
    data_ = frame_df[frame_df['platform_collapsed'] == platform_name].copy()
    frame_terms = " + ".join(narrative_frames)
    main_inds   = " + ".join(indicators)
    inter_terms = " + ".join([f"{f}:{z}" for f in narrative_frames for z in indicators])
    formula = f"{outcome} ~ {frame_terms} + {main_inds} + {inter_terms}"
    model = smf.glm(formula=formula, data=data_, family=sm.families.Binomial()).fit(cov_type='HC3')
    return model

models = {
    'truth': fit_model_with_interactions(df_post, 'truth', outcome='reinforce_rate'),
    'Bsky/Mastodon': fit_model_with_interactions(df_post, 'Bsky/Mastodon', outcome='reinforce_rate')
}

# ============ 8) 组装矩阵（主体=交互；右侧=β_frame；底部=β_group；底部 baseline 列格子=β0） ============
PLATFORM_LABEL = {'truth': 'TruthSocial', 'Bsky/Mastodon': 'Bluesky and Mastodon'}
DISPLAY_GROUPS = ['Trump+','Trump-','Biden+','Biden-']
INDICATOR_FOR  = {'Trump+':'TrumpPos','Trump-':'TrumpNeg','Biden+':'BidenPos','Biden-':'BidenNeg'}
FRAMES_ORDER   = narrative_frames[:]  # 行顺序只含 frames

def _term_group_main(ind_name: str) -> str:
    return ind_name  # 例如 'TrumpPos'

def _term_interaction(frame: str, ind_name: str) -> str:
    return f"{frame}:{ind_name}"

def lincomb(model, terms):
    """返回线性组合/单项的 (coef, pval)"""
    idx = model.params.index
    L = np.zeros(len(idx))
    for t in (terms if isinstance(terms, (list, tuple)) else [terms]):
        if t in idx:
            L[idx.get_loc(t)] += 1.0
    res = model.t_test(L)
    return float(np.asarray(res.effect).squeeze()), float(np.asarray(res.pvalue).squeeze())

def build_panel(model, plat_key):
    """
    单平台面板：
      - 主体列：各 group（交互项 β_(f×g)）
      - 右侧列：Baseline • Frame (= β_frame)
      - 返回：coef_df, pval_df, baseline_col_name, group_row (β_group), group_row_p, beta0, p0
    """
    plat_label = PLATFORM_LABEL.get(plat_key, plat_key)
    cols = [f"{plat_label} {g}" for g in DISPLAY_GROUPS]

    coef = pd.DataFrame(index=FRAMES_ORDER, columns=cols, dtype=float)
    pval = pd.DataFrame(index=FRAMES_ORDER, columns=cols, dtype=float)
    for gdisp in DISPLAY_GROUPS:
        ind = INDICATOR_FOR[gdisp]
        col = f"{plat_label} {gdisp}"
        for f in FRAMES_ORDER:
            c, p = lincomb(model, _term_interaction(f, ind))
            coef.loc[f, col] = c
            pval.loc[f, col] = p

    # 右侧 baseline（β_frame）
    right_name = f"Intercept + Frame Main Effect ({plat_label})"
    coef[right_name] = [lincomb(model, f)[0] for f in FRAMES_ORDER]
    pval[right_name] = [lincomb(model, f)[1] for f in FRAMES_ORDER]

    # 单平台的 β_group 行（先存，稍后合并为单行）
    group_row     = pd.Series(index=list(coef.columns), dtype=float)
    group_row_p   = pd.Series(index=list(pval.columns), dtype=float)
    for gdisp in DISPLAY_GROUPS:
        ind = INDICATOR_FOR[gdisp]
        c, p = lincomb(model, _term_group_main(ind))  # 仅 group effect
        group_row[f"{plat_label} {gdisp}"] = c
        group_row_p[f"{plat_label} {gdisp}"] = p
    group_row[right_name]   = np.nan  # 右下角先空，之后放 β0
    group_row_p[right_name] = np.nan

    # 取 β0（universal intercept）
    beta0, p0 = lincomb(model, "Intercept")

    return coef, pval, right_name, group_row, group_row_p, beta0, p0

# 两个平台
coef_t, p_t, right_t, group_t, group_p_t, beta0_t, p0_t = build_panel(models['truth'], 'truth')
coef_b, p_b, right_b, group_b, group_p_b, beta0_b, p0_b = build_panel(models['Bsky/Mastodon'], 'Bsky/Mastodon')

# 列方向拼接（Truth 在左，Bsky 在右）
coef_panels = pd.concat([coef_t, coef_b], axis=1)
pval_panels = pd.concat([p_t,  p_b], axis=1)

# —— 底部“单行”：Baseline • Group Main Effect（vs Neutral），并把 β0 放到各自 baseline 列的底部格 —— 
row_name = "Group Main Effect"
all_cols = list(coef_panels.columns)
bottom_coef = pd.DataFrame(index=[row_name], columns=all_cols, dtype=float)
bottom_pval = pd.DataFrame(index=[row_name], columns=all_cols, dtype=float)

# 填入各 panel 的 β_group
bottom_coef.loc[row_name, group_t.index]   = group_t.values
bottom_pval.loc[row_name, group_p_t.index] = group_p_t.values
bottom_coef.loc[row_name, group_b.index]   = group_b.values
bottom_pval.loc[row_name, group_p_b.index] = group_p_b.values

# 把 β0 放到 baseline frame 列的底部格
bottom_coef.loc[row_name, right_t] = beta0_t
bottom_pval.loc[row_name, right_t] = p0_t
bottom_coef.loc[row_name, right_b] = beta0_b
bottom_pval.loc[row_name, right_b] = p0_b

# 最终用于绘图的矩阵（行：frames + 单行 group-effect baseline）
coef_plot = pd.concat([coef_panels.loc[FRAMES_ORDER], bottom_coef], axis=0)
pval_plot = pd.concat([pval_panels.loc[FRAMES_ORDER], bottom_pval], axis=0)

# ============ 9) 可视化（窄白条分隔 panel & baseline 列；底部单行） ============
alpha = 0.05
signif_mask = (pval_plot < alpha)
annot = coef_plot.applymap(lambda v: "" if pd.isna(v) else f"{v:.2f}")

plt.figure(figsize=(16, 10), dpi=300)

# 灰底
    # 可选：若想让非显著更“白”，也可把一个复制版设成 NaN（这里按你的风格保留）
masked_data = coef_plot.copy()
masked_data[~signif_mask] = np.nan  # 若你用它作底层，即可真正把非显著涂白
ax = sns.heatmap(
    coef_plot, cmap="Greys", cbar=False,
    annot=annot, fmt="", linewidths=0.4, linecolor='lightgray',
    annot_kws = {"size": 12, "color": "white"}
)
# 仅显著叠色
sns.heatmap(
    coef_plot, cmap="coolwarm", center=0,
    mask=~signif_mask, annot=False,
    cbar_kws={"label": "Logit Coefficient (significant only)"},
    linewidths=0.4, linecolor='lightgray',
    vmin=-0.3, vmax=0.3
)

# 窄白分隔线
cols = list(coef_plot.columns)
x_truth_baseline = cols.index(right_t)           # Truth baseline 列左边界
x_bsky_baseline  = cols.index(right_b)           # Bsky  baseline 列左边界
x_panel_split    = cols.index(right_t) + 1       # 两个 panel 分界（在 Truth baseline 之后）
y_baseline       = len(FRAMES_ORDER)             # frames 与底部单行之间

for x in [x_truth_baseline, x_panel_split, x_bsky_baseline]:
    if x is x_panel_split:
        ax.vlines(x, *ax.get_ylim(), colors="white", linewidth=12.0, zorder=10,)
    ax.vlines(x, *ax.get_ylim(), colors="white", linewidth=8.0, zorder=10,)
ax.hlines(y_baseline, *ax.get_xlim(), colors="white", linewidth=3.0, zorder=10)

# 标签与 panel 标题
ax.set_yticklabels([lbl.get_text().replace("_", " ") for lbl in ax.get_yticklabels()], fontweight = 'bold')
plt.xticks(rotation=45, ha='right', fontweight = 'bold')

def panel_center_x(panel_cols):
    xs = [cols.index(c) + 0.5 for c in panel_cols]
    return (min(xs) + max(xs)) / 2

truth_panel_cols = [c for c in cols if c.startswith(PLATFORM_LABEL['truth'])] + [right_t]
bsky_panel_cols  = [c for c in cols if c.startswith(PLATFORM_LABEL['Bsky/Mastodon'])] + [right_b]

ax.text(panel_center_x(truth_panel_cols), -0.6, PLATFORM_LABEL['truth'],
        ha='center', va='bottom', fontsize=14, fontweight='bold', transform=ax.transData)
ax.text(panel_center_x(bsky_panel_cols), -0.6, PLATFORM_LABEL['Bsky/Mastodon'],
        ha='center', va='bottom', fontsize=14, fontweight='bold', transform=ax.transData)

plt.ylabel("Frames  +  Group Main Effect (vs Neutral)", fontsize=12, fontweight='bold')
plt.xlabel("Targets per panel  +  Frame Main Effect (per panel)", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
df.groupby("platform")[['trump']].value_counts()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# —— 1) 准备：得到各平台 × {Trump,Biden} × {positive,negative,neutral,NA} 的占比 —— 
CAT_ORDER = ["positive", "negative", "neutral"]  # 需要其它取值会自动追加到最后
PALETTE = {
    "positive": "#55a868",  # 绿
    "negative": "#c44e52",  # 红
    "neutral":  "#dd8452",  # 灰
    "did not mention":       "#4c72b0",  # 缺失（若有）
}

def _counts_share(df, platform_col, target_col, categories=CAT_ORDER):
    s = df[target_col].fillna("NA")
    counts = (
        df.assign(**{target_col: s})
          .groupby(platform_col)[target_col]
          .value_counts()
          .unstack(fill_value=0)
    )
    # 确保列顺序稳定
    extra = [c for c in counts.columns if c not in categories + (["NA"] if "NA" in counts.columns else [])]
    order = categories + extra + (["NA"] if "NA" in counts.columns and "NA" not in categories else [])
    counts = counts.reindex(columns=order, fill_value=0)
    share = counts.div(counts.sum(axis=1), axis=0).fillna(0) * 100
    return counts.astype(int), share, order


trump_counts, trump_share, trump_order = _counts_share(
    df.assign(trump=df['trump'].fillna("did not mention")),
    "platform", "trump", CAT_ORDER
)

biden_counts, biden_share, biden_order = _counts_share(
    df.assign(biden=df['biden'].fillna("did not mention")),
    "platform", "biden", CAT_ORDER
)

order = list(dict.fromkeys(trump_order + biden_order))  # 合并顺序

# （可选）查看表格
# display(trump_counts); display((trump_share.round(1)))
# display(biden_counts); display((biden_share.round(1)))

# —— 2) 画图：并列的 100% 堆叠柱（每个平台两根柱：Trump & Biden）——
platforms = list(trump_share.index.union(biden_share.index))
x = np.arange(len(platforms))
bar_w = 0.38

fig, ax = plt.subplots(figsize=(max(6, 1.5*len(platforms)), 4.8), dpi=500)

# 准备底部累计（两个并列柱各自一套 bottom）
bottom_T = np.zeros(len(platforms))
bottom_B = np.zeros(len(platforms))

# 取对应平台顺序的矩阵（没有的平台补 0）
T = trump_share.reindex(index=platforms, columns=order, fill_value=0)
B = biden_share.reindex(index=platforms, columns=order, fill_value=0)

for cat in order:
    color = PALETTE.get(cat, None)
    # Trump（左柱）
    ax.bar(x - bar_w/2, T[cat].values, width=bar_w, bottom=bottom_T, color=color, edgecolor="white", linewidth=0.5, label=cat if cat==order[0] else None)
    # Biden（右柱）
    ax.bar(x + bar_w/2, B[cat].values, width=bar_w, bottom=bottom_B, color=color, edgecolor="white", linewidth=0.5)
    bottom_T += T[cat].values
    bottom_B += B[cat].values
# —— 二级 X 轴标签：下层 = Trump/Biden；更下层 = 平台名 —— 

# 1) 下层（主轴）的小标签：Trump / Biden
minor_locs   = np.ravel(np.column_stack([x - bar_w/2, x + bar_w/2]))  # 每个平台两根柱的位置
minor_labels = (["Trump", "Biden"] * len(platforms))

ax.set_xticks(minor_locs)
ax.set_xticklabels(minor_labels, fontsize=8)
ax.tick_params(axis="x", which="major", bottom=False)  # 暂时关掉主轴的大刻度线

# 2) 更下层的大标签：平台名（使用 secondary_xaxis 叠一层底部轴）
ax2 = ax.secondary_xaxis('bottom', functions=(lambda v: v, lambda v: v))
ax2.set_xticks(x)
platform_map = {
    'bluesky' : 'Bluesky',
    'truth'  : 'Truth Social',
    'mastodon': 'Mastodon',
}
platforms = [platform_map.get(p, p) for p in platforms]
ax2.set_xticklabels(platforms, fontsize=11, fontweight="bold")
ax2.tick_params(axis="x", pad=22, length=0)  # pad 调大，让平台名在更底下一排；不画刻度线
ax2.spines["bottom"].set_visible(False)

# 3) 适当留白，防止两行文字挤到图外
plt.subplots_adjust(bottom=0.22)
# 百分比标注（>5% 再标，避免太密）
def annotate_stack(values, bottoms, xpos):
    for i, (v, b) in enumerate(zip(values, bottoms)):
        if v >= 2:
            ax.text(xpos[i], b - v/2, f"{v:.0f}%", ha="center", va="center", color="white", fontsize=9)

# 对每个类别单独标注需要拿到分段高度，这里简单地只标注“占比最大的段”
max_T = T.idxmax(axis=1)
max_B = B.idxmax(axis=1)

import matplotlib.patheffects as pe

def annotate_all(ax, mat, xpos, shift, order, threshold=5, color="white", fontsize=9):
    """
    在 100% 堆叠柱上为每段标注百分比（仅标注 >= threshold 的段）
    - mat: 平台×类别 的占比（单位：百分数 0~100），DataFrame
    - xpos: 平台中心的 x 坐标数组（与你画柱子的 x 一样）
    - shift: Trump 用 -bar_w/2，Biden 用 +bar_w/2
    - order: 类别顺序（与堆叠顺序一致）
    """
    platforms = mat.index.tolist()
    for i, p in enumerate(platforms):
        row = mat.loc[p, order].fillna(0)
        csum = row.cumsum()
        prev = csum.shift(fill_value=0)
        for cat in order:
            h = float(row[cat])        # 该段高度（百分比）
            if h >= threshold:
                y = float(prev[cat] + h/2)
                txt = ax.text(xpos[i] + shift, y, f"{h:.0f}%",
                              ha="center", va="center", color=color, fontsize=fontsize)
                # 白字加黑描边，浅色段也清晰（可删）
                txt.set_path_effects([pe.Stroke(linewidth=1.4, foreground='black'), pe.Normal()])

# 调用：T、B 分别是你用于作图的 trump_share、biden_share（百分比 0~100）
annotate_all(ax, T, x, shift=-bar_w/2, order=order, threshold=5, color="white", fontsize=9)
annotate_all(ax, B, x, shift=+bar_w/2, order=order, threshold=5, color="white", fontsize=9)



# 轴与图例
#ax.set_xticks(x, platforms)
ax.set_ylim(0, 100)
ax.set_ylabel("Share (%)")
#ax.set_title("Trump vs Biden attitude distribution by platform (side-by-side, 100% stacked)")
# 构造图例（按 order 显示）
handles = [plt.Rectangle((0,0),1,1,color=PALETTE.get(cat, None)) for cat in order]
ax.legend(handles, order, title="Attitude", bbox_to_anchor=(1.02,1), loc="upper left", frameon=False)
ax.grid(axis="y", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()



# Case Study

In [ ]:
narrative_analysis_df.loc[(narrative_analysis_df['trump_trial']) | (narrative_analysis_df['hunter_biden_trial']) | (narrative_analysis_df['presidential_election_debate'])].drop_duplicates(subset=['post_id']).groupby('platform')[['trump_trial', 'hunter_biden_trial', 'presidential_election_debate']].sum()

In [ ]:
narrative_analysis_df

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 假设 df 已经加载
# 事件字段
df = narrative_analysis_df.drop_duplicates(subset=['post_id'])
event_cols = ["trump_trial", "hunter_biden_trial", "presidential_election_debate"]

# narrative frame 列
frame_cols = [
    "Persecution", "Corruption", "Accountability", "Irony/Detachment",
    "Heroism", "Civic Critique", "Moral Decay", "Media Manipulation",
    "Strategic Pragmatism", "Cultural Identity"
]

# case study 结果存放
results = {}

for event in event_cols:
    # 选出该事件的post
    subset = df[df[event] == True]

    # 平台 × frame 平均值
    frame_means = subset.groupby("platform")[frame_cols].mean()

    # 存结果
    results[event] = frame_means

    # 简单可视化 (radar 或 bar chart)
    fig, ax = plt.subplots(figsize=(10, 6), dpi=300)
    frame_means.T.plot(kind="bar", ax=ax)
    ax.set_title(f"Narrative Frame Distribution – {event.replace('_', ' ').title()}")
    ax.set_ylabel("Average Frame Intensity")
    ax.set_xlabel("Narrative Frames")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    ax.legend(title="Platform")
    plt.tight_layout()
    plt.show()

# engagement 对比 (可选)
for event in event_cols:
    subset = df[df[event] == 1]
    eng = subset.groupby("platform")[["replies_count", "reblogs_count"]].mean()
    print(f"\nEvent: {event}")
    print(eng)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
# ======================
# 0) 数据准备
# ======================
# 去重
df = narrative_analysis_df.drop_duplicates(subset=['post_id']).copy()

# 事件字段
event_cols = ["trump_trial", "hunter_biden_trial", "presidential_election_debate"]

# narrative frame 列
frame_cols = [
    "Persecution","Corruption","Accountability","Irony/Detachment",
    "Heroism","Civic Critique","Moral Decay","Media Manipulation",
    "Strategic Pragmatism","Cultural Identity"
]

# 统一把事件列转为布尔值，兼容 True/1/0
for c in event_cols:
    df[c] = df[c].astype(bool)

# 计算：Event × Platform × Frame 的平均强度
records = []
for event in event_cols:
    sub = df[df[event]]
    if sub.empty:
        continue
    m = sub.groupby("platform")[frame_cols].mean()
    m["Event"] = event
    m = m.reset_index().rename(columns={"platform":"Platform"})
    records.append(m.melt(id_vars=["Platform","Event"], value_vars=frame_cols,
                          var_name="Frame", value_name="Intensity"))

long_df = pd.concat(records, ignore_index=True)
# 事件名美化
event_name_map = {
    "trump_trial": "Trump Trial",
    "hunter_biden_trial": "Hunter Biden Trial",
    "presidential_election_debate": "Presidential Election Debate"
}
long_df["Event"] = long_df["Event"].map(event_name_map)

# 框架按整体均值排序，便于扫读
frame_order = (long_df.groupby("Frame")["Intensity"]
                        .mean().sort_values(ascending=False).index.tolist())
platform_order = sorted(long_df["Platform"].unique().tolist())
event_order = ["Trump Trial","Hunter Biden Trial","Presidential Election Debate"]

# 统一量程（便于跨图比较）
vmin, vmax = long_df["Intensity"].min(), long_df["Intensity"].max()

# ======================
# 1) 三平台并排热力图（强烈推荐）
# ======================
def heatmap(data, row_labels, col_labels, ax=None,
            cbar_kw=None, cbarlabel="", **kwargs):
    """
    Create a heatmap from a numpy array and two lists of labels.

    Parameters
    ----------
    data
        A 2D numpy array of shape (M, N).
    row_labels
        A list or array of length M with the labels for the rows.
    col_labels
        A list or array of length N with the labels for the columns.
    ax
        A `matplotlib.axes.Axes` instance to which the heatmap is plotted.  If
        not provided, use current Axes or create a new one.  Optional.
    cbar_kw
        A dictionary with arguments to `matplotlib.Figure.colorbar`.  Optional.
    cbarlabel
        The label for the colorbar.  Optional.
    **kwargs
        All other arguments are forwarded to `imshow`.
    """

    if ax is None:
        ax = plt.gca()

    if cbar_kw is None:
        cbar_kw = {}

    # Plot the heatmap
    im = ax.imshow(data, **kwargs)

    # Create colorbar
    cbar = ax.figure.colorbar(im, ax=ax, **cbar_kw)
    cbar.ax.set_ylabel(cbarlabel, rotation=-90, va="bottom")

    # Show all ticks and label them with the respective list entries.
    ax.set_xticks(range(data.shape[1]), labels=col_labels,
                  rotation=-30, ha="right", rotation_mode="anchor")
    ax.set_yticks(range(data.shape[0]), labels=row_labels)

    # Let the horizontal axes labeling appear on top.
    ax.tick_params(top=True, bottom=False,
                   labeltop=True, labelbottom=False)

    # Turn spines off and create white grid.
    ax.spines[:].set_visible(False)

    ax.set_xticks(np.arange(data.shape[1]+1)-.5, minor=True)
    ax.set_yticks(np.arange(data.shape[0]+1)-.5, minor=True)
    ax.grid(which="minor", color="w", linestyle='-', linewidth=3)
    ax.tick_params(which="minor", bottom=False, left=False)

    return im, cbar


def annotate_heatmap(im, data=None, valfmt="{x:.2f}",
                     textcolors=("black", "white"),
                     threshold=None, **textkw):
    """
    A function to annotate a heatmap.

    Parameters
    ----------
    im
        The AxesImage to be labeled.
    data
        Data used to annotate.  If None, the image's data is used.  Optional.
    valfmt
        The format of the annotations inside the heatmap.  This should either
        use the string format method, e.g. "$ {x:.2f}", or be a
        `matplotlib.ticker.Formatter`.  Optional.
    textcolors
        A pair of colors.  The first is used for values below a threshold,
        the second for those above.  Optional.
    threshold
        Value in data units according to which the colors from textcolors are
        applied.  If None (the default) uses the middle of the colormap as
        separation.  Optional.
    **kwargs
        All other arguments are forwarded to each call to `text` used to create
        the text labels.
    """

    if not isinstance(data, (list, np.ndarray)):
        data = im.get_array()

    # Normalize the threshold to the images color range.
    if threshold is not None:
        threshold = im.norm(threshold)
    else:
        threshold = im.norm(data.max())/2.

    # Set default alignment to center, but allow it to be
    # overwritten by textkw.
    kw = dict(horizontalalignment="center",
              verticalalignment="center")
    kw.update(textkw)

    # Get the formatter in case a string is supplied
    if isinstance(valfmt, str):
        valfmt = matplotlib.ticker.StrMethodFormatter(valfmt)

    # Loop over the data and create a `Text` for each "pixel".
    # Change the text's color depending on the data.
    texts = []
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            kw.update(color=textcolors[int(im.norm(data[i, j]) > threshold)])
            text = im.axes.text(j, i, valfmt(data[i, j], None), **kw)
            texts.append(text)

    return texts

fig, axes = plt.subplots(1, 3, figsize=(16, 7), sharey=True)
for ax, plat in zip(axes, platform_order):
    mat = (long_df[long_df["Platform"]==plat]
           .pivot_table(index="Frame", columns="Event", values="Intensity")
           .reindex(index=frame_order, columns=event_order))
    im = heatmap(harvest, vegetables, farmers, ax=ax,
                   cmap="YlGn", cbarlabel="harvest [t/year]")
    ax.set_title(plat)
    ax.set_xticks(range(len(event_order))); ax.set_xticklabels(event_order, rotation=30, ha="right")
    ax.set_yticks(range(len(frame_order))); ax.set_yticklabels(frame_order)
    # 在格子中标注数值
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat.iat[i,j]
            if pd.notna(val):
                ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=9)

fig.subplots_adjust(right=0.88)
cax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
fig.colorbar(im, cax=cax, label="Average Frame Intensity")
plt.suptitle("Frame Intensity by Platform (rows=Frames, cols=Events)", y=0.98, fontsize=14)
plt.show()

# ======================
# 2) 斜率小倍数图（每个框架一个面板，看三事件趋势 + 平台对比）
# ======================
frames = frame_order
n = len(frames)
cols = 3
rows = int(np.ceil(n/cols))
fig, axes = plt.subplots(rows, cols, figsize=(18, 5*rows), sharey=True)
axes = axes.ravel()

for k, fr in enumerate(frames):
    ax = axes[k]
    sub = long_df[long_df["Frame"]==fr]
    # 画每个平台在三个事件上的变化
    for plat in platform_order:
        s = (sub[sub["Platform"]==plat]
             .set_index("Event").reindex(event_order)["Intensity"])
        ax.plot(event_order, s.values, marker="o", label=plat)
        # 在两端标注
        if pd.notna(s.iloc[0]):
            ax.text(0, s.iloc[0], f"{s.iloc[0]:.2f}", va="bottom", fontsize=9)
        if pd.notna(s.iloc[-1]):
            ax.text(len(event_order)-1, s.iloc[-1], f"{s.iloc[-1]:.2f}", va="bottom", fontsize=9)
    ax.set_title(fr)
    ax.grid(axis="y", linestyle=":", alpha=.4)
for i in range(n, rows*cols):
    fig.delaxes(axes[i])
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=len(platform_order), frameon=False)
plt.suptitle("Per-Frame Trends Across Events (lines=Platforms)", y=0.98, fontsize=14)
plt.tight_layout(rect=[0,0,1,0.95])
plt.show()

# ======================
# 3) Facet 柱状图（并排三事件，统一量程）
# ======================
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
for ax, ev in zip(axes, event_order):
    sub = (long_df[long_df["Event"]==ev]
           .pivot_table(index="Frame", columns="Platform", values="Intensity")
           .reindex(index=frame_order, columns=platform_order))
    sub.plot(kind="bar", ax=ax, width=0.85)
    ax.set_title(ev)
    ax.set_xlabel("Narrative Frames")
    ax.set_ylabel("Average Frame Intensity")
    ax.set_xticklabels(sub.index, rotation=45, ha="right")
    ax.grid(axis="y", linestyle=":", alpha=.4)
    ax.legend().remove()
# 统一图例
handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles, labels, title="Platform", loc="upper center", ncol=len(platform_order), frameon=False)
plt.suptitle("Narrative Frame Distribution — All Events Side-by-Side", y=0.98, fontsize=14)
plt.tight_layout(rect=[0,0,1,0.93])
plt.show()

# ======================
# 4) Engagement 对比（沿用你的打印逻辑，修正布尔筛选）
# ======================
for ev_raw, ev_pretty in event_name_map.items():
    sub = df[df[ev_raw]]
    if sub.empty:
        print(f"\nEvent: {ev_pretty} — No posts")
        continue
    eng = sub.groupby("platform")[["replies_count","reblogs_count"]].mean(numeric_only=True)
    print(f"\nEvent: {ev_pretty}")
    print(eng.round(2))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
def heatmap(data, row_labels, col_labels, ax=None,
            cbar_kw=None, cbarlabel="", show_cbar=True, **kwargs):
    import matplotlib.pyplot as plt
    import numpy as np

    if ax is None:
        ax = plt.gca()
    if cbar_kw is None:
        cbar_kw = {}

    im = ax.imshow(data, **kwargs,)

    # 只在需要时画色条
    # cbar = None
    # if show_cbar:
    #     cbar = ax.figure.colorbar(im, ax=ax, **cbar_kw)
    #     cbar.ax.set_ylabel(cbarlabel, rotation=-90, va="bottom")

    # 底部显示 x 标签，避免和标题冲突
    ax.set_xticks(range(data.shape[1]), labels=col_labels, rotation=30, ha="right")
    ax.set_yticks(range(data.shape[0]), labels=row_labels)
    ax.tick_params(top=False, bottom=True, labeltop=False, labelbottom=True)

    # 去边框+白色网格
    ax.spines[:].set_visible(False)
    ax.set_xticks(np.arange(data.shape[1]+1)-.5, minor=True)
    ax.set_yticks(np.arange(data.shape[0]+1)-.5, minor=True)
    ax.grid(which="minor", color="w", linestyle='-', linewidth=3)
    ax.tick_params(which="minor", bottom=False, left=False)

    return im, cbar


def annotate_heatmap(im, data=None, valfmt="{x:.2f}",
                     textcolors=("black", "white"),
                     threshold=None, **textkw):
    """
    A function to annotate a heatmap.

    Parameters
    ----------
    im
        The AxesImage to be labeled.
    data
        Data used to annotate.  If None, the image's data is used.  Optional.
    valfmt
        The format of the annotations inside the heatmap.  This should either
        use the string format method, e.g. "$ {x:.2f}", or be a
        `matplotlib.ticker.Formatter`.  Optional.
    textcolors
        A pair of colors.  The first is used for values below a threshold,
        the second for those above.  Optional.
    threshold
        Value in data units according to which the colors from textcolors are
        applied.  If None (the default) uses the middle of the colormap as
        separation.  Optional.
    **kwargs
        All other arguments are forwarded to each call to `text` used to create
        the text labels.
    """

    if not isinstance(data, (list, np.ndarray)):
        data = im.get_array()

    # Normalize the threshold to the images color range.
    if threshold is not None:
        threshold = im.norm(threshold)
    else:
        threshold = im.norm(data.max())/2.

    # Set default alignment to center, but allow it to be
    # overwritten by textkw.
    kw = dict(horizontalalignment="center",
              verticalalignment="center")
    kw.update(textkw)

    # Get the formatter in case a string is supplied
    if isinstance(valfmt, str):
        valfmt = matplotlib.ticker.StrMethodFormatter(valfmt)

    # Loop over the data and create a `Text` for each "pixel".
    # Change the text's color depending on the data.
    texts = []
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            kw.update(color=textcolors[int(im.norm(data[i, j]) > threshold)])
            text = im.axes.text(j, i, valfmt(data[i, j], None), **kw)
            texts.append(text)

    return texts
# ----------------------------------------------------
# 假设 long_df 已经在上游准备好：
# 列: ["Platform","Event","Frame","Intensity"]
# 如果你还没有 long_df，可复用你之前的计算逻辑得到它。
# ----------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

platforms   = ["bluesky", "mastodon", "truth"]
event_order = ["Trump Trial", "Hunter Biden Trial", "Presidential Election Debate"]
frame_order = (long_df.groupby("Frame")["Intensity"]
                        .mean().sort_values(ascending=False).index.tolist())

vmin, vmax = long_df["Intensity"].min(), long_df["Intensity"].max()

fig, axes = plt.subplots(
    1, 3,
    figsize=(12, 6), dpi=300,
    sharey=True,
    gridspec_kw={'wspace': 0.05}
)

ims = []
for i, (ax, plat) in enumerate(zip(axes, platforms)):
    pivot = (long_df[long_df["Platform"] == plat]
             .pivot_table(index="Frame", columns="Event", values="Intensity")
             .reindex(index=frame_order, columns=event_order))

    im = sns.heatmap(
        pivot, ax=ax,
        cmap="YlGn", vmin=vmin, vmax=vmax,
        annot=True, fmt=".2f",
        cbar=False,         # 关键：子图不画色条
        square=True,
        linewidths=0, linecolor=None
    )

    ax.set_title(plat, fontsize=12)
    ax.tick_params(axis="x", rotation=30)


    ims.append(im.collections[0])  # heatmap 的 QuadMesh

# 用 fig.colorbar 统一挂一个色条
cbar = fig.colorbar(ims[0], ax=axes, orientation="vertical",
                    fraction=0.025, pad=0.02)
cbar.set_label("Average Frame Intensity")

fig.suptitle("Frame Intensity by Platform (rows = Frames, cols = Events)", fontsize=16, y=0.98)
plt.tight_layout()
plt.show()




In [ ]:
import seaborn as sns
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

platforms   = ["bluesky", "mastodon", "truth"]
platforms_to_name = {"bluesky": "Bluesky", "mastodon": "Mastodon", "truth": "Truth Social"}
event_order = ["Trump Trial", "Hunter Biden Trial", "Presidential Election Debate"]
frame_order = (long_df.groupby("Frame")["Intensity"]
                        .mean().sort_values(ascending=False).index.tolist())
vmin, vmax = long_df["Intensity"].min(), long_df["Intensity"].max()

# -------- 关键：按单元格大小计算 figsize --------
R = len(frame_order)      # 行数（Frame）
C = len(event_order)      # 列数（Event）
A = len(platforms)        # 子图个数

cell = 0.45               # 每个单元格的边长（英寸），可微调 0.40~0.55
W = A * C * cell + 1.1    # 图宽（含色条与左右边距）
H = R * cell + 1.4        # 图高（含底部刻度与标题）

fig, axes = plt.subplots(
    1, A, figsize=(W, H), dpi=600, sharey=True,
    gridspec_kw={'wspace': 0.02}   # 子图之间几乎贴紧
)
plt.subplots_adjust(left=0.12, right=0.94, top=0.90, bottom=0.18, wspace=0.02)

ims = []

for i, (ax, plat) in enumerate(zip(axes, platforms)):
    pivot = (long_df[long_df["Platform"] == plat]
             .pivot_table(index="Frame", columns="Event", values="Intensity", aggfunc="mean")
             .reindex(index=frame_order, columns=event_order))
    

    pivot = pivot.div(pivot.sum(axis=0), axis=1)

    im = sns.heatmap(
        pivot, ax=ax,
        cmap="YlGn",   # 分布范围固定在 [0,1]
        annot=True, fmt=".2f",
        cbar=False,
        square=True,
        linewidths=0
    )

    ax.set_title(platforms_to_name[plat], fontsize=12, pad=6)
    ax.tick_params(axis="x", rotation=90, pad=2)
    ax.tick_params(axis="y", pad=2)
    if i > 0:
        ax.set_ylabel(None)
    ax.set_xlabel("Event" if i == 1 else "")
    ims.append(im.collections[0])

cbar = fig.colorbar(
    ims[0],
    ax=axes,
    orientation="vertical",
    fraction=0.03,
    pad=0.04,

)
cbar.set_label("Frame Intensity Distribution", rotation=90, labelpad=8)



#fig.suptitle("Frame Intensity by Platform (rows = Frames, cols = Events)", fontsize=16, y=0.98)
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

# 事件列与可读名字
EVENT_COLS = ["trump_trial", "hunter_biden_trial", "presidential_election_debate"]
EVENT_NAME = {
    "trump_trial": "Trump Trial",
    "hunter_biden_trial": "Hunter Biden Trial",
    "presidential_election_debate": "Presidential Debate",
}

# 类别顺序与配色
CAT_ORDER = ["positive", "negative", "neutral", "did not mention"]
PALETTE = {
    "positive": "#55a868",
    "negative": "#c44e52",
    "neutral":  "#dd8452",
    "did not mention": "#4c72b0",
}

# 统一把态度列缺失填“did not mention”
df_plot = df.copy()
df_plot["trump"] = df_plot["trump"].fillna("did not mention")
df_plot["biden"] = df_plot["biden"].fillna("did not mention")

def counts_share(sub_df, platform_col, target_col, categories):
    # 统计：平台 × 态度
    counts = (sub_df.groupby(platform_col)[target_col]
                    .value_counts().unstack(fill_value=0))
    # 稳定列顺序
    for c in categories:
        if c not in counts.columns:
            counts[c] = 0
    counts = counts.reindex(columns=categories, fill_value=0)
    share = counts.div(counts.sum(axis=1), axis=0).fillna(0) * 100
    return share

# 平台显示名
PLATFORM_MAP = {'bluesky': 'Bluesky', 'mastodon': 'Mastodon', 'truth': 'Truth Social'}

# 画图（每个事件一个子图）
fig, axes = plt.subplots(1, 3, figsize=(14, 5.3), dpi=300, sharey=True,
                         gridspec_kw={'wspace': 0.12})

bar_w = 0.38
for ax, ev in zip(axes, EVENT_COLS):
    sub = df_plot[df_plot[ev].astype(bool)]
    if sub.empty:
        ax.text(0.5, 0.5, "No posts", ha="center", va="center", fontsize=11)
        ax.set_axis_off()
        continue

    # Trump / Biden 的占比矩阵（index=平台, columns=态度类别）
    T = counts_share(sub, "platform", "trump", CAT_ORDER)
    B = counts_share(sub, "platform", "biden", CAT_ORDER)

    # 平台顺序统一（出现的就画，缺的补 0）
    platforms = sorted(set(T.index).union(B.index))
    x = np.arange(len(platforms))
    T = T.reindex(index=platforms, columns=CAT_ORDER, fill_value=0)
    B = B.reindex(index=platforms, columns=CAT_ORDER, fill_value=0)

    # 两组 100% 堆叠柱
    bottom_T = np.zeros(len(platforms))
    bottom_B = np.zeros(len(platforms))
    for j, cat in enumerate(CAT_ORDER):
        color = PALETTE.get(cat, None)
        ax.bar(x - bar_w/2, T[cat].values, width=bar_w,
               bottom=bottom_T, color=color, edgecolor="white", linewidth=0.5,
               label=cat if (ev == EVENT_COLS[0] and j == 0) else None)  # 只为第一幅首段挂图例句柄
        ax.bar(x + bar_w/2, B[cat].values, width=bar_w,
               bottom=bottom_B, color=color, edgecolor="white", linewidth=0.5)
        bottom_T += T[cat].values
        bottom_B += B[cat].values

    # 百分比标注（≥5%才标），白字+黑描边以确保可读
    def annotate(mat, shift):
        csum = mat.cumsum(axis=1)
        prev = csum.shift(axis=1, fill_value=0)
        for i, p in enumerate(mat.index):
            for cat in CAT_ORDER:
                h = float(mat.loc[p, cat])
                if h >= 5:
                    y = float(prev.loc[p, cat] + h/2)
                    txt = ax.text(x[i] + shift, y, f"{h:.0f}%", ha="center", va="center",
                                  color="white", fontsize=9)
                    txt.set_path_effects([pe.Stroke(linewidth=1.4, foreground='black'), pe.Normal()])
    annotate(T, -bar_w/2)
    annotate(B, +bar_w/2)

    # x 轴标签：每个平台两根柱，次标签 Trump/Biden
    ax.set_xticks(np.ravel(np.column_stack([x - bar_w/2, x + bar_w/2])))
    ax.set_xticklabels(["Trump", "Biden"] * len(platforms), fontsize=8)
    ax.tick_params(axis="x", which="major", bottom=False)

    # 加一层平台名（次级 x 轴）
    ax2 = ax.secondary_xaxis('bottom', functions=(lambda v: v, lambda v: v))
    ax2.set_xticks(x)
    ax2.set_xticklabels([PLATFORM_MAP.get(p, p) for p in platforms],
                        fontsize=10, fontweight="bold")
    ax2.tick_params(axis="x", pad=22, length=0)
    ax2.spines["bottom"].set_visible(False)

    ax.set_ylim(0, 100)
    ax.set_ylabel("Share (%)" if ax is axes[0] else "")
    ax.set_title(EVENT_NAME[ev], fontsize=12)
    ax.grid(axis="y", linestyle="--", alpha=0.3)

# 统一图例（只放一份）
handles = [plt.Rectangle((0,0),1,1,color=PALETTE[c]) for c in CAT_ORDER]
fig.legend(handles, CAT_ORDER, title="Attitude",
           bbox_to_anchor=(1.05, 0.98), loc="upper right", frameon=False)

#fig.suptitle("Attitudes toward Trump vs. Biden by Platform within Each Event", y=0.99)
plt.tight_layout(rect=[0,0,0.98,0.95])
plt.show()


# Validation

In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import krippendorff

df = pd.read_csv("../data/data/NDIR_sampled_200.csv")

# ===== 工具函数 =====
def alpha_nominal(coders: dict):
    """Krippendorff α (nominal)，支持多 coder，自动类别编码"""
    vals = set()
    for s in coders.values():
        vals |= set(s.dropna().astype(str).tolist())
    categories = sorted(list(vals))
    cat2id = {c: i for i, c in enumerate(categories)}
    value_domain = list(range(len(categories)))
    data = []
    for s in coders.values():
        codes = s.astype(str).map(cat2id)
        arr = codes.astype("Float64").to_numpy()
        data.append(arr.astype(float))
    data = np.vstack(data)
    return float(krippendorff.alpha(
        reliability_data=data,
        level_of_measurement="nominal",
        value_domain=value_domain
    ))

def run_alpha(group_name, coders):
    """返回四种对比结果"""
    results = []
    names = list(coders.keys())
    # 两两
    if len(names) >= 2:
        for i in range(len(names)):
            for j in range(i+1, len(names)):
                pair = {names[i]: coders[names[i]], names[j]: coders[names[j]]}
                a = alpha_nominal(pair)
                results.append((group_name, f"{names[i]}_vs_{names[j]}", a, pair[names[i]].notna().sum()))
    # 全部
    if len(names) > 2:
        a_all = alpha_nominal(coders)
        results.append((group_name, "all_three", a_all, min(len(s.dropna()) for s in coders.values())))
    return results

# ===== Frame + Reply =====
soft_cols = [
    "Persecution","Corruption","Accountability","Irony/Detachment","Heroism",
    "Civic Critique","Moral Decay","Media Manipulation",
    "Strategic Pragmatism","Cultural Identity"
]
llm_frame = np.array(soft_cols)[df[soft_cols].astype(float).values.argmax(axis=1)]
reply_col = "classification" if "classification" in df.columns else "prediction"

coders_frame = {
    "human1": pd.concat([df["Frame_human"]], ignore_index=True),
    "human2": pd.concat([df["Frame_human2"]], ignore_index=True),
    "llm": pd.concat([pd.Series(llm_frame)], ignore_index=True)
}
res_frame = run_alpha("Frame", coders_frame)

# Reply
coder_reply = {
    "human1": pd.concat([df["reply_human"].str.lower()], ignore_index=True),
    "human2": pd.concat([df["reply_human2"].str.lower()], ignore_index=True),
    "llm": pd.concat([df[reply_col].str.lower()], ignore_index=True)
}
res_reply = run_alpha("Reply", coder_reply)
# ===== Actors =====
coders_actor = {
    "human1": pd.concat([df["trump_human"], df["biden_human"]], ignore_index=True),
    "human2": pd.concat([df["trump_human2"], df["biden_human2"]], ignore_index=True),
    "llm": pd.concat([df["trump"], df["biden"]], ignore_index=True)
}
res_actor = run_alpha("Actors", coders_actor)

# ===== Events (补 FALSE) =====
coders_event = {
    "human1": pd.concat([
        df["trump_trial_human"].fillna(False),
        df["hunter_biden_trial_human"].fillna(False),
        df["presidential_election_debate_human"].fillna(False)
    ], ignore_index=True),
    "human2": pd.concat([
        df["trump_trial_human2"].fillna(False),
        df["hunter_biden_trial_human2"].fillna(False),
        df["presidential_election_debate_human2"].fillna(False)
    ], ignore_index=True),
    "llm": pd.concat([
        df["trump_trial"].fillna(False),
        df["hunter_biden_trial"].fillna(False),
        df["presidential_election_debate"].fillna(False)
    ], ignore_index=True)
}
res_event = run_alpha("Events", coders_event)

# ===== 汇总 =====
results = res_frame + res_reply + res_actor + res_event
res_df = pd.DataFrame(results, columns=["Group","Comparison","Alpha","N"])
print(res_df.to_string(index=False))
res_df.to_csv("alpha_summary_grouped.csv", index=False)


In [ ]:
df_trump = df.loc[df['trump_trial'],:]
df_biden = df.loc[df['hunter_biden_trial'],:]
df_debate = df.loc[df['presidential_election_debate'],:]

In [ ]:
# summary

In [ ]:
from vllm import LLM, SamplingParams
import pandas as pd
import numpy as np
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'  # adjust based on your GPU setup
# ===== 0) Prepare data =====
# Expect df columns: ["post_id","platform","text","trump_trial","hunter_biden_trial","presidential_election_debate",
#                     "likes_count","reblogs_count","replies_count"]

EVENTS = {
    "trump_trial": "Trump Trial",
    "hunter_biden_trial": "Hunter Biden Trial",
    "presidential_election_debate": "Presidential Election Debate",
}
PLATFORMS = ["bluesky", "mastodon", "truth"]

def engagement_score(row):
    return float(row.get("likes_count",0) + row.get("reblogs_count",0) + row.get("replies_count",0))


# ===== 1) Initialize LLM =====
MODEL_ID = "google/gemma-3-27b-it"  # replace with your local repo name
llm = LLM(model=MODEL_ID, tensor_parallel_size=2)

sampling = SamplingParams(temperature=0.2, top_p=0.9, max_tokens=1400)




In [ ]:
from transformers import AutoTokenizer
from string import Template
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
SYSTEM = (
    "You are a neutral assistant. "
    "Summarize only based on provided posts. "
    "Do not speculate. If insufficient evidence, say 'insufficient_evidence'."
)
def sample_group(sub: pd.DataFrame, n=120, seed=42):
    if sub.empty: return sub
    s = sub.copy()
    s["score"] = s.apply(engagement_score, axis=1)
    top = s.sort_values("score", ascending=False).head(int(n*0.6))
    rest = s.drop(top.index)
    np.random.seed(seed)
    if len(rest) > n-len(top):
        rand = rest
    else:
        rand = rest.sample(n=max(0, n-len(top)), replace=False) if len(rest)>0 else rest
    return pd.concat([top, rand]).drop(columns="score")
def pack_posts_block(df_sub: pd.DataFrame, max_chars=18000):
    rows, total = [], 0
    for _, r in df_sub.iterrows():
        text_clean = str(r["post_full_content"]).replace("\n", " ")
        line = f"{r['post_id']} - [{text_clean}]"
        if total + len(line) > max_chars: break
        rows.append(line); total += len(line)+1
    return "\n".join(rows)
USER_TMPL = """You will summarize social posts for ONE group.

Group:
- platform: {platform}
- event: {event}

Task:
1) Identify the most representative opinions expressed in this group.
   - Capture both pro- and anti- perspectives (Trump and Biden) if they exist.
   - Summarize in 2–4 concise bullet points per perspective.
   - Roleplay: If you were a typical user in this group, write 2–3 sentences of what you might say, in the group’s tone. 
2) Provide 2–3 short direct quotes (≤30 words) that best illustrate each perspective. Include post_id.
3) Note if any opinion seems dominant (majority view) or if the group is divided.
4) Mention notable accounts or sources cited.

Return ONLY JSON with this schema:
{{
  "platform": "{platform}",
  "event": "{event}",
  "opinions": [
    {{
      "stance": "pro-trump | anti-trump | pro-biden | anti-biden | mixed | unclear",
      "summary": "Concise description of the viewpoint",
      "Roleplay": "<roleplay>",
      "quotes": [{{"post_id":"...", "text":"..."}}]
    }}
  ],
  "dominant_view": "short note",
  "notables": []
}}

Posts (each line = post_id ␞ text):
{posts_block}
"""

def build_prompt(platform, event_code, df_group):
    posts_block = pack_posts_block(sample_group(df_group, n=500))
    return [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": USER_TMPL.format(
            platform=platform,
            event=EVENTS[event_code],
            posts_block=posts_block
        )}
    ]

def pack_posts_block(df_sub: pd.DataFrame, max_chars=18000):
    rows, total = [], 0
    for _, r in df_sub.iterrows():
        text_clean = str(r["post_full_content"]).replace("\n", " ")
        line = f"{r['post_id']} - [{text_clean}]"
        if total + len(line) > max_chars: break
        rows.append(line); total += len(line)+1
    return "\n".join(rows)
# ===== 2) Map: Summarize each (platform, event) =====
prompts, meta = [], []
for plat in PLATFORMS:
    for ev in EVENTS.keys():
        sub = df[(df["platform"]==plat) & (df[ev]==True)]
        if sub.empty: continue
        prompts.append(build_prompt(plat, ev, sub))
        meta.append((plat, ev))
prompts = [tokenizer.apply_chat_template(p, tokenize=False) for p in prompts]
outputs = llm.generate(prompts, sampling)

summaries = []
for (plat, ev), out in zip(meta, outputs):
    summaries.append({
        "platform": plat,
        "event": ev,
        "json": out.outputs[0].text.strip()
    })

# ===== 3) Reduce (optional): Merge by platform =====
REDUCE_TMPL = Template("""You are given three JSON summaries for the same platform across different events.
Synthesize an overall summary. Return JSON:
{
  "platform": "$platform",
  "overall_themes": ["..."],
  "stance_overview": {"trump":"", "biden":""},
  "differences_by_event": [{"event":"...", "key_changes":["..."]}],
  "open_questions": ["..."]
}

Input:
$json_block
""")
reduce_prompts, reduce_meta = [], []
for plat in PLATFORMS:
    js = [s["json"] for s in summaries if s["platform"]==plat]
    if not js: continue
    block = "\n---\n".join(js)
    reduce_prompts.append([
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": REDUCE_TMPL.substitute(platform=plat, json_block=block)}
    ])

    reduce_meta.append(plat)

reduce_sampling = SamplingParams(temperature=0.2, top_p=0.9, max_tokens=1200)
reduce_prompts = tokenizer.apply_chat_template(reduce_prompts, tokenize=False)
reduce_out = llm.generate(reduce_prompts, reduce_sampling)

platform_overviews = {
    plat: out.outputs[0].text.strip()
    for plat, out in zip(reduce_meta, reduce_out)
}


In [ ]:
# group_summarizer.py
# -*- coding: utf-8 -*-

from __future__ import annotations
from typing import Dict, Iterable, List, Tuple, Any
from string import Template
import os
import numpy as np
import pandas as pd

# ---- vLLM / HF
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

# ===============================
# 0) Environment & Data Settings
# ===============================
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'  # adjust based on your GPU setup

# Expect df columns (plus stance columns 'trump','biden'):
# ["post_id","platform","text",
#  "trump_trial","hunter_biden_trial","presidential_election_debate",
#  "likes_count","reblogs_count","replies_count",
#  "trump","biden"]
#
# 'trump'/'biden' ∈ {"positive","negative", None/NaN/others}

EVENTS: Dict[str, str] = {
    "trump_trial": "Trump Trial",
    "hunter_biden_trial": "Hunter Biden Trial",
    "presidential_election_debate": "Presidential Election Debate",
}
PLATFORMS = ["bluesky", "mastodon", "truth"]

def engagement_score(row: pd.Series) -> float:
    # Your provided definition (simple sum of 3 counters)
    return float(row.get("likes_count", 0) + row.get("reblogs_count", 0) + row.get("replies_count", 0))

# ====================
# 1) Initialize the LLM
# ====================
MODEL_ID = "google/gemma-3-27b-it"  # replace with your local repo name
# Tokenizer is used to apply chat template (Gemma expects chat formatting)
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
except Exception:
    tokenizer = None

#llm = LLM(model=MODEL_ID, tensor_parallel_size=2)
#sampling = SamplingParams(temperature=0.2, top_p=0.9, max_tokens=1400)

# ====================
# 2) System instruction
# ====================
SYSTEM = (
    "You are a neutral assistant. "
    "Summarize only based on provided posts. "
    "Do not speculate. If insufficient evidence, say 'insufficient_evidence'."
)

# ====================
# 3) Sampling & packing
# ====================
def sample_group(sub: pd.DataFrame, n: int = 120, seed: int = 42) -> pd.DataFrame:
    """60% top by engagement + remaining random (without replacement)."""
    if sub is None or sub.empty:
        return sub if sub is not None else pd.DataFrame()
    s = sub.copy()
    s["score"] = s.apply(engagement_score, axis=1)
    n = max(0, int(n))
    if n == 0:
        return s.drop(columns="score", errors="ignore")

    top_k = max(1, int(n * 0.6))
    top = s.sort_values("score", ascending=False).head(top_k)
    rest = s.drop(top.index)

    np.random.seed(seed)
    need_rand = max(0, n - len(top))
    if need_rand <= 0 or rest.empty:
        out = top
    else:
        need_rand = min(need_rand, len(rest))
        rand = rest.sample(n=need_rand, replace=False, random_state=seed)
        out = pd.concat([top, rand], ignore_index=True)

    return out.drop(columns="score", errors="ignore")

def pack_posts_block(df_sub: pd.DataFrame, max_chars: int = 18000) -> str:
    """Pack as '<post_id>-[<text>]' while enforcing a char budget."""
    if df_sub is None or df_sub.empty:
        return ""
    rows, total = [], 0
    for _, r in df_sub.iterrows():
        text_clean = str(r.get("post_full_content", "")).replace("\n", " ").strip()
        pid = str(r.get("post_id", "")).strip()
        line = f"{pid} - [{text_clean}]"
        if total + len(line) > max_chars:
            break
        rows.append(line)
        total += len(line) + 1
    return "\n".join(rows)
def pack_posts_block_limited(df_sub: pd.DataFrame,
                             max_lines: int,
                             max_chars_per_line: int) -> str:
    if df_sub is None or df_sub.empty or max_lines <= 0:
        return ""
    rows = []
    for _, r in df_sub.head(max_lines).iterrows():
        text_clean = str(r.get("post_full_content", "")).replace("\n", " ").strip()
        if max_chars_per_line and len(text_clean) > max_chars_per_line:
            text_clean = text_clean[:max_chars_per_line].rstrip() + "…"
        pid = str(r.get("post_id", "")).strip()
        rows.append(f"{pid} - [{text_clean}]")
    return "\n".join(rows)

# ====================
# 4) Five-bucket split
# ====================
VALID = {"positive", "negative"}

def _norm(x: Any) -> str:
    return str(x).strip().lower()

def label_other(row: pd.Series) -> bool:
    t = _norm(row.get("trump", ""))
    b = _norm(row.get("biden", ""))
    return not (t in VALID or b in VALID)

def split_by_stance(df_sub: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Return (pro_trump, anti_trump, pro_biden, anti_biden, other)."""
    s = df_sub.copy()
    for col in ["trump", "biden"]:
        if col not in s.columns:
            s[col] = None
        s[col] = s[col].astype(str)

    pro_trump  = s[s["trump"].str.lower().eq("positive")]
    anti_trump = s[s["trump"].str.lower().eq("negative")]
    pro_biden  = s[s["biden"].str.lower().eq("positive")]
    anti_biden = s[s["biden"].str.lower().eq("negative")]
    other      = s[s.apply(label_other, axis=1)]
    return pro_trump, anti_trump, pro_biden, anti_biden, other

# ==================================
# 5) User template (five-bucket JSON)
# ==================================
USER_TMPL_GROUPED = """You will summarize social posts for ONE group.

Group:
- platform: {platform}
- event: {event}

Stance buckets (pre-filtered by me):
- Pro-Trump: posts where trump == positive
- Anti-Trump: posts where trump == negative
- Pro-Biden: posts where biden == positive
- Anti-Biden: posts where biden == negative
- Other: posts where neither trump nor biden is positive/negative

Tasks:
1) Identify representative opinions for EACH bucket that has content.
   - Give 2–4 concise bullet points per bucket.
   - Keep wording neutral and traceable to the given posts.
   - Roleplay: If you were a typical user in this group, write 2–3 sentences (overall, not per-bucket), matching the group's tone.

2) Provide 2–3 short direct quotes (≤30 words) per bucket, each with its post_id.
3) Say whether one perspective appears dominant or the group is divided, with a brief reason.
4) Mention notable accounts or external sources cited.

Constraints:
- Use ONLY the provided posts below; do not invent content.
- Every quote ≤30 words, keep original language, and include post_id.
- If a bucket has insufficient examples, write "insufficient_evidence" for its quotes/points.
- If a candidate side (Trump/Biden) doesn't appear, still keep the bucket key but mark as "not_observed".

Return ONLY JSON with this schema:
{{
  "platform": "{platform}",
  "event": "{event}",
  "summary": {{
    "trump": {{
      "positive": {{
        "summary": "Concise description of the viewpoint",
        "roleplay": "If you were a typical user in this group, write 2–3 sentences (overall, not per-bucket), matching the group's tone.",
        "bullets": ["..."],
        "quotes": [{{"post_id":"...", "text":"..."}}]
      }},
      "negative": {{
        "summary": "Concise description of the viewpoint",
        "roleplay": "If you were a typical user in this group, write 2–3 sentences (overall, not per-bucket), matching the group's tone.",
        "bullets": ["..."],
        "quotes": [{{"post_id":"...", "text":"..."}}]
      }}
    }},
    "biden": {{
      "positive": {{
        "summary": "Concise description of the viewpoint",
        "roleplay": "If you were a typical user in this group, write 2–3 sentences (overall, not per-bucket), matching the group's tone.",
        "bullets": ["..."],
        "quotes": [{{"post_id":"...", "text":"..."}}]
      }},
      "negative": {{
        "summary": "Concise description of the viewpoint",
        "roleplay": "If you were a typical user in this group, write 2–3 sentences (overall, not per-bucket), matching the group's tone.",
        "bullets": ["..."],
        "quotes": [{{"post_id":"...", "text":"..."}}]
      }}
    }},
    "other": {{
      "summary": "Concise description of the viewpoint",
      "roleplay": "If you were a typical user in this group, write 2–3 sentences (overall, not per-bucket), matching the group's tone.",
      "bullets": ["..."],
      "quotes": [{{"post_id":"...", "text":"..."}}]
    }}
  }},
  "roleplay_typical_user": "<2–3 sentences>",
  "dominant_view": "dominant | divided | unclear",
  "dominant_reason": "≤2 sentences",
  "notables": [{{"account":"@name","relation":"pro/anti/info","notes":"..."}}],
  "limitations": "1–2 sentences about data limits"
}}

Posts_ProTrump (post_id-[text]):
{block_pro_trump}

Posts_AntiTrump (post_id-[text]):
{block_anti_trump}

Posts_ProBiden (post_id-[text]):
{block_pro_biden}

Posts_AntiBiden (post_id-[text]):
{block_anti_biden}

Posts_Other (post_id-[text]):
{block_other}
"""

# ======================
# 6) Prompt construction
# ======================
def _serialize_chat(system: str, user: str) -> str:
    """
    Serialize chat using tokenizer's chat template if available; otherwise fallback.
    """
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    if tokenizer is not None and hasattr(tokenizer, "apply_chat_template"):
        try:
            return tokenizer.apply_chat_template(msgs, tokenize=False)
        except Exception:
            pass
    # Fallback: naive role-tagged prompt
    return f"<|system|>\n{system}\n\n<|user|>\n{user}"

def build_prompt_grouped(platform: str, event_code: str, df_group: pd.DataFrame) -> str:
    pro_t, anti_t, pro_b, anti_b, other = split_by_stance(df_group)

    # ---- 预算参数（可调）----
    MAX_INPUT_TOKENS = 6000      # 给输入的上限（为输出留 1400+ 的空间）
    BASE_LINES = 80              # 常规事件每桶的基础行数上限
    MIN_LINES = 20               # 每桶最少行数（有内容时）
    MAX_CHARS_PER_LINE = 240     # 每行截断
    HEAVY_EVENT_MULTIPLIER = 0.5 # 超大事件整体收紧（行数减半）
    # ---- 识别“大事件”：按该事件总样本量判定 ----
    total_rows = len(df_group)
    heavy_event = total_rows >= 10000  # trump_trial 明显会触发

    # 初步为每桶设置行数上限：有内容的桶才分配
    buckets = {
        "pro_t": pro_t,
        "anti_t": anti_t,
        "pro_b": pro_b,
        "anti_b": anti_b,
        "other": other,
    }
    active = {k: v for k, v in buckets.items() if not v.empty}
    per_bucket_lines = {}
    for k in buckets.keys():
        if k in active:
            per_bucket_lines[k] = BASE_LINES
        else:
            per_bucket_lines[k] = 0

    if heavy_event:
        for k in per_bucket_lines:
            per_bucket_lines[k] = int(max(MIN_LINES, per_bucket_lines[k] * HEAVY_EVENT_MULTIPLIER))

    # 先粗打包一次，估 token，如超限再收紧
    def pack_all(max_chars_per_line):
        blocks = {
            "block_pro_trump":  pack_posts_block_limited(pro_t,   per_bucket_lines["pro_t"],   max_chars_per_line),
            "block_anti_trump": pack_posts_block_limited(anti_t,  per_bucket_lines["anti_t"],  max_chars_per_line),
            "block_pro_biden":  pack_posts_block_limited(pro_b,   per_bucket_lines["pro_b"],   max_chars_per_line),
            "block_anti_biden": pack_posts_block_limited(anti_b,  per_bucket_lines["anti_b"],  max_chars_per_line),
            "block_other":      pack_posts_block_limited(other,   per_bucket_lines["other"],   max_chars_per_line),
        }
        user = USER_TMPL_GROUPED.format(platform=platform, event=EVENTS[event_code], **blocks)
        return _serialize_chat(SYSTEM, user)

    serialized = pack_all(MAX_CHARS_PER_LINE)
    est_tokens = estimate_tokens(serialized, tokenizer)

    # 如果仍然超预算，逐步收紧：先降 per-line chars，再降每桶行数
    step_chars = [200, 160, 120, 90, 60]
    step_lines_factor = [0.8, 0.6, 0.5, 0.4]  # 逐步缩行

    si = 0
    li = 0
    while est_tokens > MAX_INPUT_TOKENS and (si < len(step_chars) or li < len(step_lines_factor)):
        if si < len(step_chars):
            # 优先缩短每行文本
            serialized = pack_all(step_chars[si])
            est_tokens = estimate_tokens(serialized, tokenizer)
            si += 1
            continue
        if li < len(step_lines_factor):
            # 再按比例缩行（不低于 MIN_LINES）
            factor = step_lines_factor[li]
            for k in per_bucket_lines:
                if per_bucket_lines[k] > 0:
                    per_bucket_lines[k] = max(MIN_LINES, int(per_bucket_lines[k] * factor))
            serialized = pack_all(step_chars[min(si, len(step_chars)-1)] if si>0 else MAX_CHARS_PER_LINE)
            est_tokens = estimate_tokens(serialized, tokenizer)
            li += 1
            continue

    # 最终序列化
    return serialized


def build_all_prompts(df: pd.DataFrame,
                      platforms: Iterable[str],
                      events: Dict[str, str]) -> Tuple[List[str], List[Tuple[str, str]]]:
    """
    Returns serialized prompts and meta list of (platform, event_code).
    Requires df[event_code] == True boolean flags per row.
    """
    prompts_serialized, meta = [], []
    for plat in platforms:
        for ev_code in events.keys():
            sub = df[(df["platform"] == plat) & (df[ev_code] == True)]
            if sub.empty:
                continue
            prompts_serialized.append(build_prompt_grouped(plat, ev_code, sub))
            meta.append((plat, ev_code))
    return prompts_serialized, meta

# =========================
# 7) Run & collect routines
# =========================
def run_generation(prompts_serialized: List[str]) -> List[Any]:
    """Thin wrapper around vLLM.generate."""
    return llm.generate(prompts_serialized, sampling)

def collect_summaries(meta: List[Tuple[str, str]], outputs: List[Any]) -> List[Dict[str, str]]:
    """
    Standardize returned JSON text under key 'json'.
    Compatible with vLLM .outputs[0].text or simple string outputs.
    """
    results = []
    for (plat, ev), out in zip(meta, outputs):
        text = None
        if hasattr(out, "outputs") and out.outputs:
            text = getattr(out.outputs[0], "text", None)
        if text is None:
            text = str(out)
        results.append({
            "platform": plat,
            "event": ev,
            "json": (text or "").strip()
        })
    return results

# =========================
# 8) Optional: platform reducer
# =========================
REDUCE_TMPL = Template("""You are given multiple JSON summaries for the same platform across different events.
Synthesize an overall summary. Return ONLY JSON:
{
  "platform": "$platform",
  "overall_themes": ["..."],
  "stance_overview": {
    "trump": {"positive":"...", "negative":"..."},
    "biden": {"positive":"...", "negative":"..."},
    "other": "..."
  },
  "differences_by_event": [{"event":"...", "key_changes":["..."]}],
  "open_questions": ["..."]
}

Input:
$json_block
""")

def build_reduce_prompts(summaries: List[Dict[str, str]],
                         platforms: Iterable[str]) -> Tuple[List[str], List[str]]:
    prompts_serialized, meta = [], []
    for plat in platforms:
        js = [s["json"] for s in summaries if s["platform"] == plat and s.get("json")]
        if not js:
            continue
        block = "\n---\n".join(js)
        user = REDUCE_TMPL.substitute(platform=plat, json_block=block)
        prompts_serialized.append(_serialize_chat(SYSTEM, user))
        meta.append(plat)
    return prompts_serialized, meta

def collect_platform_overviews(platforms_meta: List[str], reduce_outputs: List[Any]) -> Dict[str, str]:
    overviews = {}
    for plat, out in zip(platforms_meta, reduce_outputs):
        text = None
        if hasattr(out, "outputs") and out.outputs:
            text = getattr(out.outputs[0], "text", None)
        if text is None:
            text = str(out)
        overviews[plat] = (text or "").strip()
    return overviews

# =========================
# 9) Example main (optional)
# =========================
if __name__ == "__main__":
    # Example dummy df (replace with your real data)
    df = narrative_analysis_df.copy()

    prompts_serialized, meta = build_all_prompts(df, PLATFORMS, EVENTS)
    # NOTE: running generation requires the model weights locally available.
    outputs = run_generation(prompts_serialized)
    summaries = collect_summaries(meta, outputs)
    reduce_prompts, reduce_meta = build_reduce_prompts(summaries, PLATFORMS)
    reduce_outputs = run_generation(reduce_prompts)
    platform_overviews = collect_platform_overviews(reduce_meta, reduce_outputs)
    print("Wired up. Plug in real df and uncomment generation calls.")


In [ ]:
df['post_full_content'].isna().sum()

In [ ]:
from pprint import pprint
pprint(summaries)

In [ ]:
pprint(platform_overviews)

In [ ]:
def debug_event_slice(df, platforms, events):
    print("== Event slice check ==")
    for ev in events.keys():
        mask_bool  = df[ev]==True if ev in df.columns else pd.Series([False]*len(df))
        mask_truth = df[ev].astype(bool) if ev in df.columns else pd.Series([False]*len(df))
        mask_name  = (df["event"]==ev) if "event" in df.columns else pd.Series([False]*len(df))
        print(f"\nEvent {ev}:")
        print("  bool True    :", mask_bool.sum()  if ev in df else "N/A")
        print("  truthy       :", mask_truth.sum() if ev in df else "N/A")
        print("  event==ev    :", mask_name.sum()  if "event" in df.columns else "N/A")
        for plat in platforms:
            sub = df[(df["platform"]==plat) & (mask_truth if ev in df else mask_name)]
            print(f"    {plat:<10} rows:", len(sub))


In [ ]:
debug_event_slice(df, PLATFORMS, EVENTS)

In [ ]:
# group_summarizer.py
# -*- coding: utf-8 -*-

from __future__ import annotations
from typing import Dict, Iterable, List, Tuple, Any
from string import Template
import os
import json
import numpy as np
import pandas as pd

# ---- vLLM / HF
try:
    from vllm import LLM, SamplingParams
except Exception:
    LLM = None
    SamplingParams = None

from transformers import AutoTokenizer

# ===============================
# 0) Environment & Data Settings
# ===============================
# Adjust based on your GPU setup
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0,1')

# Expected DataFrame columns (plus stance columns 'trump','biden'):
# ["post_id","platform","text","post_full_content",
#  "trump_trial","hunter_biden_trial","presidential_election_debate",
#  "likes_count","reblogs_count","replies_count",
#  "trump","biden"]
#
# 'trump'/'biden' ∈ {"positive","negative", None/NaN/others}

EVENTS: Dict[str, str] = {
    "trump_trial": "Trump Trial",
    "hunter_biden_trial": "Hunter Biden Trial",
    "presidential_election_debate": "Presidential Election Debate",
}
PLATFORMS = ["bluesky", "mastodon", "truth"]

# Per-bucket post cap for inputs to the model
MAX_POSTS_PER_BUCKET = 800

def engagement_score(row: pd.Series) -> float:
    """Simple engagement score = likes + reblogs + replies."""
    return float(row.get("likes_count", 0) + row.get("reblogs_count", 0) + row.get("replies_count", 0))

# ====================
# 1) Initialize the LLM
# ====================
MODEL_ID = "google/gemma-3-27b-it"  # replace with your local repo name
# Tokenizer is used to apply chat template (Gemma expects chat formatting)
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
except Exception:
    tokenizer = None

# Uncomment these once your environment has vLLM + the model weights ready:
# llm = LLM(model=MODEL_ID, tensor_parallel_size=2)
# sampling = SamplingParams(temperature=0.2, top_p=0.9, max_tokens=1400)

# ====================
# 2) System instruction
# ====================
SYSTEM = (
    "You are a neutral assistant. "
    "Summarize only based on provided posts. "
    "Do not speculate. If insufficient evidence, say 'insufficient_evidence'."
)

# ====================
# Utility: token estimate
# ====================
def estimate_tokens(text: str, tok: Any = None) -> int:
    """
    Roughly estimate token count for an input string.
    Prefer tokenizer.encode; otherwise approximate by len/4.
    """
    if tok is not None:
        try:
            return len(tok.encode(text))
        except Exception:
            pass
    return max(1, int(len(text) / 4))

# ====================
# 3) Sampling & packing
# ====================
def sample_group(sub: pd.DataFrame, n: int = 120, seed: int = 42) -> pd.DataFrame:
    """
    60% by engagement (top-k) + remaining random (without replacement).
    """
    if sub is None or sub.empty:
        return sub if sub is not None else pd.DataFrame()
    s = sub.copy()
    s["score"] = s.apply(engagement_score, axis=1)
    n = max(0, int(n))
    if n == 0:
        return s.drop(columns="score", errors="ignore")

    top_k = max(1, int(n * 0.6))
    top = s.sort_values("score", ascending=False).head(top_k)
    rest = s.drop(top.index)

    np.random.seed(seed)
    need_rand = max(0, n - len(top))
    if need_rand <= 0 or rest.empty:
        out = top
    else:
        need_rand = min(need_rand, len(rest))
        rand = rest.sample(n=need_rand, replace=False, random_state=seed)
        out = pd.concat([top, rand], ignore_index=True)

    return out.drop(columns="score", errors="ignore")

def pack_posts_block(df_sub: pd.DataFrame, max_chars: int = 18000) -> str:
    """
    Pack rows as '<post_id> - [<text>]' under a total character budget.
    """
    if df_sub is None or df_sub.empty:
        return ""
    rows, total = [], 0
    for _, r in df_sub.iterrows():
        text_clean = str(r.get("post_full_content", "")).replace("\n", " ").strip()
        pid = str(r.get("post_id", "")).strip()
        line = f"{pid} - [{text_clean}]"
        if total + len(line) > max_chars:
            break
        rows.append(line)
        total += len(line) + 1
    return "\n".join(rows)

def pack_posts_block_limited(df_sub: pd.DataFrame,
                             max_lines: int,
                             max_chars_per_line: int) -> str:
    """
    Pack up to max_lines rows; each line trimmed to max_chars_per_line.
    """
    if df_sub is None or df_sub.empty or max_lines <= 0:
        return ""
    rows = []
    for _, r in df_sub.head(max_lines).iterrows():
        text_clean = str(r.get("post_full_content", "")).replace("\n", " ").strip()
        if max_chars_per_line and len(text_clean) > max_chars_per_line:
            text_clean = text_clean[:max_chars_per_line].rstrip() + "…"
        pid = str(r.get("post_id", "")).strip()
        rows.append(f"{pid} - [{text_clean}]")
    return "\n".join(rows)

# ====================
# 4) Five-bucket split
# ====================
VALID = {"positive", "negative"}

def _norm(x: Any) -> str:
    return str(x).strip().lower()

def label_other(row: pd.Series) -> bool:
    t = _norm(row.get("trump", ""))
    b = _norm(row.get("biden", ""))
    return not (t in VALID or b in VALID)

def split_by_stance(df_sub: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Return (pro_trump, anti_trump, pro_biden, anti_biden, other).
    """
    s = df_sub.copy()
    for col in ["trump", "biden"]:
        if col not in s.columns:
            s[col] = None
        s[col] = s[col].astype(str)

    pro_trump  = s[s["trump"].str.lower().eq("positive")]
    anti_trump = s[s["trump"].str.lower().eq("negative")]
    pro_biden  = s[s["biden"].str.lower().eq("positive")]
    anti_biden = s[s["biden"].str.lower().eq("negative")]
    other      = s[s.apply(label_other, axis=1)]
    return pro_trump, anti_trump, pro_biden, anti_biden, other

# ==================================
# 5) Original "grouped" (five buckets) template — kept for optional use
# ==================================
USER_TMPL_GROUPED = """You will summarize social posts for ONE group.

Group:
- platform: {platform}
- event: {event}

Stance buckets (pre-filtered by me):
- Pro-Trump: posts where trump == positive
- Anti-Trump: posts where trump == negative
- Pro-Biden: posts where biden == positive
- Anti-Biden: posts where biden == negative
- Other: posts where neither trump nor biden is positive/negative

Tasks:
1) Identify representative opinions for EACH bucket that has content.
   - Give 2–4 concise bullet points per bucket.
   - Keep wording neutral and traceable to the given posts.
   - Roleplay: If you were a typical user in this group, write 2–3 sentences (overall, not per-bucket), matching the group's tone.

2) Provide 2–3 short direct quotes (≤30 words) per bucket, each with its post_id.
3) Say whether one perspective appears dominant or the group is divided, with a brief reason.
4) Mention notable accounts or external sources cited.

Constraints:
- Use ONLY the provided posts below; do not invent content.
- Every quote ≤30 words, keep original language, and include post_id.
- If a bucket has insufficient examples, write "insufficient_evidence" for its quotes/points.
- If a candidate side (Trump/Biden) doesn't appear, still keep the bucket key but mark as "not_observed".

Return ONLY JSON with this schema:
{
  "platform": "{platform}",
  "event": "{event}",
  "summary": {
    "trump": {
      "positive": {
        "summary": "Concise description of the viewpoint",
        "roleplay": "If you were a typical user in this group, write 2–3 sentences (overall, not per-bucket), matching the group's tone.",
        "bullets": ["..."],
        "quotes": [{"post_id":"...", "text":"..."}]
      },
      "negative": {
        "summary": "Concise description of the viewpoint",
        "roleplay": "If you were a typical user in this group, write 2–3 sentences (overall, not per-bucket), matching the group's tone.",
        "bullets": ["..."],
        "quotes": [{"post_id":"...", "text":"..."}]
      }
    },
    "biden": {
      "positive": {
        "summary": "Concise description of the viewpoint",
        "roleplay": "If you were a typical user in this group, write 2–3 sentences (overall, not per-bucket), matching the group's tone.",
        "bullets": ["..."],
        "quotes": [{"post_id":"...", "text":"..."}]
      },
      "negative": {
        "summary": "Concise description of the viewpoint",
        "roleplay": "If you were a typical user in this group, write 2–3 sentences (overall, not per-bucket), matching the group's tone.",
        "bullets": ["..."],
        "quotes": [{"post_id":"...", "text":"..."}]
      }
    },
    "other": {
      "summary": "Concise description of the viewpoint",
      "roleplay": "If you were a typical user in this group, write 2–3 sentences (overall, not per-bucket), matching the group's tone.",
      "bullets": ["..."],
      "quotes": [{"post_id":"...", "text":"..."}]
    }
  },
  "roleplay_typical_user": "<2–3 sentences>",
  "dominant_view": "dominant | divided | unclear",
  "dominant_reason": "≤2 sentences",
  "notables": [{"account":"@name","relation":"pro/anti/info","notes":"..."}],
  "limitations": "1–2 sentences about data limits"
}

Posts_ProTrump (post_id-[text]):
{block_pro_trump}

Posts_AntiTrump (post_id-[text]):
{block_anti_trump}

Posts_ProBiden (post_id-[text]):
{block_pro_biden}

Posts_AntiBiden (post_id-[text]):
{block_anti_biden}

Posts_Other (post_id-[text]):
{block_other}
"""

# ======================
# 5*) Single-bucket template (one bucket per prompt)
# ======================
USER_TMPL_SINGLE_BUCKET = """You will summarize social posts for ONE attitude bucket.

Group Context:
- platform: {platform}
- event: {event}
- bucket: {bucket_readable}

Tasks (for THIS bucket only):
1) Identify representative opinions.
   - Give 2–4 concise bullet points.
   - Keep wording neutral and traceable to the given posts.
2) Provide 2–3 short direct quotes (≤30 words).
3) Roleplay: If you were a typical user in this group, write 2–3 sentences matching the group's tone.

Constraints:
- Use ONLY the provided posts; do not invent content.
- Every quote ≤30 words, keep original language, and include post_id.
- If this bucket has insufficient examples, write "insufficient_evidence".

Return ONLY JSON with this schema:
{{
  "platform": "{platform}",
  "event": "{event}",
  "bucket": "{bucket_key}",
  "summary": {{
    "bucket_overview": "Concise description of the viewpoint",
    "bullets": ["..."],
    "quotes": ["text"],
    "roleplay": "If you were a typical member of this group, write 2–3 sentences in the group’s tone that capture the overall summary of the bucket you provided earlier."
  }}
}}

Posts ([text]):
{posts_block}
"""

# ======================
# 6) Prompt construction (shared)
# ======================
def _serialize_chat(system: str, user: str) -> str:
    """
    Serialize messages using tokenizer's chat template if available;
    otherwise fall back to simple role-tagged prompt.
    """
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    if tokenizer is not None and hasattr(tokenizer, "apply_chat_template"):
        try:
            return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        except Exception:
            pass
    return f"<|system|>\n{system}\n\n<|user|>\n{user}"

# ======================
# 6*) "Grouped" prompt builder (kept for optional use)
# ======================
def build_prompt_grouped(platform: str, event_code: str, df_group: pd.DataFrame) -> str:
    pro_t, anti_t, pro_b, anti_b, other = split_by_stance(df_group)

    # ---- Budget parameters (tune as needed) ----
    MAX_INPUT_TOKENS = 50000
    BASE_LINES = 80
    MIN_LINES = 20
    MAX_CHARS_PER_LINE = 240
    HEAVY_EVENT_MULTIPLIER = 0.5

    total_rows = len(df_group)
    heavy_event = total_rows >= 10000

    buckets = {
        "pro_t": pro_t,
        "anti_t": anti_t,
        "pro_b": pro_b,
        "anti_b": anti_b,
        "other": other,
    }
    active = {k: v for k, v in buckets.items() if not v.empty}
    per_bucket_lines = {}
    for k in buckets.keys():
        per_bucket_lines[k] = BASE_LINES if k in active else 0

    if heavy_event:
        for k in per_bucket_lines:
            per_bucket_lines[k] = int(max(MIN_LINES, per_bucket_lines[k] * HEAVY_EVENT_MULTIPLIER))

    def pack_all(max_chars_per_line):
        blocks = {
            "block_pro_trump":  pack_posts_block_limited(pro_t,   per_bucket_lines["pro_t"],   max_chars_per_line),
            "block_anti_trump": pack_posts_block_limited(anti_t,  per_bucket_lines["anti_t"],  max_chars_per_line),
            "block_pro_biden":  pack_posts_block_limited(pro_b,   per_bucket_lines["pro_b"],   max_chars_per_line),
            "block_anti_biden": pack_posts_block_limited(anti_b,  per_bucket_lines["anti_b"],  max_chars_per_line),
            "block_other":      pack_posts_block_limited(other,   per_bucket_lines["other"],   max_chars_per_line),
        }
        user = USER_TMPL_GROUPED.format(platform=platform, event=EVENTS[event_code], **blocks)
        return _serialize_chat(SYSTEM, user)

    serialized = pack_all(MAX_CHARS_PER_LINE)
    est_tokens = estimate_tokens(serialized, tokenizer)

    step_chars = [200, 160, 120, 90, 60]
    step_lines_factor = [0.8, 0.6, 0.5, 0.4]

    si = 0
    li = 0
    while est_tokens > MAX_INPUT_TOKENS and (si < len(step_chars) or li < len(step_lines_factor)):
        if si < len(step_chars):
            serialized = pack_all(step_chars[si])
            est_tokens = estimate_tokens(serialized, tokenizer)
            si += 1
            continue
        if li < len(step_lines_factor):
            factor = step_lines_factor[li]
            for k in per_bucket_lines:
                if per_bucket_lines[k] > 0:
                    per_bucket_lines[k] = max(MIN_LINES, int(per_bucket_lines[k] * factor))
            serialized = pack_all(step_chars[min(si, len(step_chars)-1)] if si > 0 else MAX_CHARS_PER_LINE)
            est_tokens = estimate_tokens(serialized, tokenizer)
            li += 1
            continue

    return serialized

# ======================
# 5**) Bucket name mapping (single-bucket mode)
# ======================
BUCKET_READABLE = {
    "pro_trump":  "Pro-Trump (trump == positive)",
    "anti_trump": "Anti-Trump (trump == negative)",
    "pro_biden":  "Pro-Biden (biden == positive)",
    "anti_biden": "Anti-Biden (biden == negative)",
    "other":      "Other (neither trump nor biden marked positive/negative)"
}

# ======================
# 5***) Cap + sample per bucket (single-bucket mode)
# ======================
def cap_and_sample(df_sub: pd.DataFrame, n_max: int = MAX_POSTS_PER_BUCKET, seed: int = 1015) -> pd.DataFrame:
    """
    Apply engagement+random sampling up to n_max rows.
    """
    if df_sub is None or df_sub.empty:
        return df_sub if df_sub is not None else pd.DataFrame()
    n = min(int(n_max), len(df_sub))
    return sample_group(df_sub, n=n, seed=seed)

def pack_posts_block_by_count(df_sub: pd.DataFrame,
                              max_count: int = MAX_POSTS_PER_BUCKET,
                              max_chars_per_line: int = 240) -> str:
    """
    Pack strictly up to max_count posts; each line optionally truncated
    to max_chars_per_line to avoid overlong inputs.
    """
    if df_sub is None or df_sub.empty:
        return ""
    rows = []
    for _, r in df_sub.head(max_count).iterrows():
        text_clean = str(r.get("post_full_content", "")).replace("\n", " ").strip()
        if max_chars_per_line and len(text_clean) > max_chars_per_line:
            text_clean = text_clean[:max_chars_per_line].rstrip() + "…"
        rows.append(f"[{text_clean}]")
    return "\n".join(rows)

# ======================
# 6*) Build single-bucket prompt
# ======================
def build_prompt_single_bucket(platform: str,
                               event_code: str,
                               bucket_key: str,
                               df_group: pd.DataFrame) -> str:
    """
    bucket_key ∈ {"pro_trump","anti_trump","pro_biden","anti_biden","other"}
    df_group: DataFrame filtered to the target bucket.
    """
    df_bucket = cap_and_sample(df_group, n_max=MAX_POSTS_PER_BUCKET)
    posts_block = pack_posts_block_by_count(df_bucket,
                                            max_count=MAX_POSTS_PER_BUCKET,
                                            max_chars_per_line=240)

    user = USER_TMPL_SINGLE_BUCKET.format(
        platform=platform,
        event=EVENTS[event_code],
        bucket_readable=BUCKET_READABLE[bucket_key],
        bucket_key=bucket_key,
        posts_block=posts_block
    )
    return _serialize_chat(SYSTEM, user)

# ======================
# 6**) Split buckets and return prompts (single-bucket mode)
# ======================
def split_buckets(df_sub: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    pro_t, anti_t, pro_b, anti_b, other = split_by_stance(df_sub)
    return {
        "pro_trump":  pro_t,
        "anti_trump": anti_t,
        "pro_biden":  pro_b,
        "anti_biden": anti_b,
        "other":      other
    }

def build_all_prompts(df: pd.DataFrame,
                      platforms: Iterable[str],
                      events: Dict[str, str]) -> Tuple[List[str], List[Tuple[str, str, str]]]:
    """
    Single-bucket mode:
    Returns (prompts_serialized, meta)
      - meta: (platform, event_code, bucket_key)
      - one prompt per non-empty bucket (each bucket ≤ MAX_POSTS_PER_BUCKET)
    Requires df[event_code] == True when event column exists.
    """
    prompts_serialized, meta = [], []
    for plat in platforms:
        for ev_code in events.keys():
            mask = (df["platform"] == plat)
            if ev_code in df.columns:
                mask = mask & (df[ev_code] == True)
            sub = df[mask]
            if sub.empty:
                continue
            buckets = split_buckets(sub)
            for bkey, bdf in buckets.items():
                if bdf is None or bdf.empty:
                    continue
                prompts_serialized.append(build_prompt_single_bucket(plat, ev_code, bkey, bdf))
                meta.append((plat, ev_code, bkey))
    return prompts_serialized, meta

# =========================
# 7) Run & collect routines
# =========================
def run_generation(prompts_serialized: List[str], sampling: SamplingParams) -> List[Any]:
    """
    Thin wrapper around vLLM.generate. Make sure `llm` and `sampling`
    are instantiated before calling this.
    """
    if 'llm' not in globals() or globals().get('llm') is None:
        raise RuntimeError("LLM is not initialized. Instantiate llm = LLM(...), sampling = SamplingParams(...) first.")
    return llm.generate(prompts_serialized, sampling)

def collect_summaries(meta: List[Tuple[str, str, str]], outputs: List[Any]) -> List[Dict[str, str]]:
    """
    Normalize outputs; compatible with vLLM .outputs[0].text or plain strings.
    Includes platform/event/bucket in the result for downstream use.
    """
    results = []
    for (plat, ev, bucket), out in zip(meta, outputs):
        text = None
        if hasattr(out, "outputs") and getattr(out, "outputs"):
            text = getattr(out.outputs[0], "text", None)
        if text is None:
            text = str(out)
        results.append({
            "platform": plat,
            "event": ev,
            "bucket": bucket,
            "json": (text or "").strip()
        })
    return results

# =========================
# 8) Platform-level reducer (for single-bucket JSON inputs)
# =========================
REDUCE_TMPL_SINGLE_BUCKETS = Template("""You are given multiple JSON summaries for the same platform across different events and stance buckets.
Synthesize an overall summary. Return ONLY JSON:
{
  "platform": "$platform",
  "overall_themes": ["..."],
  "stance_overview": {
    "trump": {"positive":"...", "negative":"..."},
    "biden": {"positive":"...", "negative":"..."},
    "other": "..."
  },
  "differences_by_event": [{"event":"...", "key_changes":["..."]}],
  "open_questions": ["..."]
}

Input JSONs (one per line):
$json_lines
""")

def build_reduce_prompts(summaries: List[Dict[str, str]],
                         platforms: Iterable[str]) -> Tuple[List[str], List[str]]:
    """
    Build reducer prompts from the single-bucket summaries produced earlier.
    Concatenate multiple JSON lines per platform to reduce prompt tokens.
    """
    prompts_serialized, meta = [], []
    for plat in platforms:
        js = [s["json"] for s in summaries if s["platform"] == plat and s.get("json")]
        if not js:
            continue
        block = "\n".join(js)
        user = REDUCE_TMPL_SINGLE_BUCKETS.substitute(platform=plat, json_lines=block)
        prompts_serialized.append(_serialize_chat(SYSTEM, user))
        meta.append(plat)
    return prompts_serialized, meta

def collect_platform_overviews(platforms_meta: List[str], reduce_outputs: List[Any]) -> Dict[str, str]:
    """
    Normalize reducer outputs into a dictionary: {platform: json_text}.
    """
    overviews = {}
    for plat, out in zip(platforms_meta, reduce_outputs):
        text = None
        if hasattr(out, "outputs") and getattr(out, "outputs"):
            text = getattr(out.outputs[0], "text", None)
        if text is None:
            text = str(out)
        overviews[plat] = (text or "").strip()
    return overviews

# =========================
# 9) Example main (optional)
# =========================
if __name__ == "__main__":
    # Example placeholder DataFrame (replace with your real data)
    
    df = narrative_analysis_df.copy()
    df['text'] = df['post_full_content'].fillna('')
    # Build prompts (single-bucket mode; platform × event × bucket; each bucket ≤ MAX_POSTS_PER_BUCKET)
    prompts_serialized, meta = build_all_prompts(df, PLATFORMS, EVENTS)
    #print(f"[Debug] prompts={len(prompts_serialized)}, meta sample={meta[:3] if meta else []}")

    # To actually run generation, uncomment and ensure vLLM is set up:
    #llm = LLM(model=MODEL_ID, tensor_parallel_size=2, max_model_len=50000)
    sampling = SamplingParams(temperature=0.2, top_p=0.9, max_tokens=600)
    outputs = run_generation(prompts_serialized, sampling)
    summaries = collect_summaries(meta, outputs)
    #
    # reduce_prompts, reduce_meta = build_reduce_prompts(summaries, PLATFORMS)
    # reduce_outputs = run_generation(reduce_prompts)
    # platform_overviews = collect_platform_overviews(reduce_meta, reduce_outputs)
    # print(json.dumps(platform_overviews, ensure_ascii=False, indent=2))

    print("Wired up. Replace the example DataFrame with your real data and uncomment vLLM calls to run.")



In [ ]:
from pprint import pprint
pprint(outputs[8].outputs)  # 查看第一个提示的内容

In [ ]:
df[(df['biden'] == 'negative')&(df['platform']=='bluesky') & (df['hunter_biden_trial']==True)]

In [ ]:
# save summaries to a JSONL file
import json
with open('summaries.jsonl', 'w', encoding='utf-8') as f:
    for item in summaries:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')
    f.write(json.dumps(item, ensure_ascii=False) + '\n')